
# ED_Pipeline_v4

**Scope:** ED workflow coordination & logistics core (QR tracking, reservations/TTL, maintenance/cleaning, SOP access, troponin trend display, serial troponin/ECG compliance, language support, sepsis-hold alerts).  
**Safe defaults:** learning mode = `shadow`, suggestions-only (no diagnoses), age/sex excluded.

**Quick start:** run the top `CONFIG` cell, then:
```python
ui_seed_and_snapshot(); show_status_board()
sop_search("sepsis bundle")
```


In [ ]:

# Writable path resolver for Kaggle/localhost
import os
def _choose_data_root():
    candidates = ["/kaggle/working", "/kaggle/temp", "/mnt/data", "."]
    for p in candidates:
        try:
            os.makedirs(p, exist_ok=True)
            test = os.path.join(p, ".__w")
            with open(test, "w") as f: f.write("ok")
            os.remove(test)
            return p
        except Exception:
            continue
    return "."
try:
    DATA_ROOT
except NameError:
    DATA_ROOT = _choose_data_root()
print("DATA_ROOT:", DATA_ROOT)


In [ ]:

# --- Global CONFIG guard (prevents NameError in Kaggle runs) ---
try:
    CONFIG
except NameError:
    CONFIG = {}
# Minimal defaults used across the notebook; safely merged by setdefault() later
CONFIG.setdefault("learning", {
    "mode": "shadow",
    "min_evidence": 20,
    "ci_level": 0.90,
    "baseline_accept": 0.60,
    "exploration_floor": 0.20,
    "max_p_delta": 0.10,
})
CONFIG.setdefault("troponin", {"delta_abs_ngL": 5.0, "delta_pct": 20.0})
CONFIG.setdefault("seed", 42)



# ED Workflow Coordination — Modular Architecture (Skills + Critics + Controller + Evolver)

This notebook **replaces the prior monolithic model** with an **agentic, modular architecture**:

- **Layer 0: Contracts** — hard safety rails (rate limits, cooldowns, budgets).
- **Layer 1: Skill library** — small, interpretable rules/models that propose actions.
- **Layer 2: Critics** — tiny calibrated heads predicting acceptance, burden, benefit.
- **Layer 3: Meta-controller** — multi-objective chooser with bounded exploration (Thompson / UCB).
- **Layer 4: Memory** — casebook of (context, action, outcome) for few-shot generalization.
- **Evolver** — weekly policy search over simulation / replay (Pareto against acceptance, fatigue, time saved).

> This is research code (not production). It runs on simulated or replay logs; when real logs are available, keep the same APIs.


In [ ]:

# --- Setup & imports
import os, json, math, random, time, warnings
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
ART = Path("./artifacts_modular"); ART.mkdir(exist_ok=True, parents=True)
RNG = np.random.default_rng(4242)


## Layer 0 — Contracts & Constraints

In [ ]:

@dataclass
class Constraints:
    # Budgets (fraction rejected tolerated after warm-up)
    fp_budget_per_h: Dict[str, float] = field(default_factory=lambda: {
        "ORDER_ECG": 0.15,
        "PAGE_CARDIOLOGY": 0.10,
        "FOLLOW_UP_IMAGING": 0.12,
        "EXPEDITE_LABS": 0.12,
        "REQUEST_BED": 0.08,
    })
    # Role rate limits (seconds between suggestions)
    per_role_rate_limit: Dict[str, int] = field(default_factory=lambda: {
        "nurse": 6*60, "resident": 5*60, "ED_MD": 4*60
    })
    # Action cooldowns (seconds)
    action_cooldown_s: Dict[str, int] = field(default_factory=lambda: {
        "PAGE_CARDIOLOGY": 20*60,
        "FOLLOW_UP_IMAGING": 30*60
    })
    max_pages_per_2h: int = 10

class ConstraintState:
    def __init__(self):
        self.last_action_ts = {}      # (enc_id, action)->ts
        self.pages_in_window = []     # timestamps
        self.last_by_role = {}        # role->ts
        self.fp_counter = {}          # action->{accepted,rejected}
    def allow(self, now_ts: float, role: str, enc_id: str, action: str, C: Constraints) -> bool:
        last_role_ts = self.last_by_role.get(role, 0.0)
        if now_ts - last_role_ts < C.per_role_rate_limit.get(role, 300):
            return False
        key = (enc_id, action)
        cooldown = C.action_cooldown_s.get(action, 0)
        last_ts = self.last_action_ts.get(key, 0.0)
        if now_ts - last_ts < cooldown:
            return False
        window_start = now_ts - 2*3600
        self.pages_in_window = [t for t in self.pages_in_window if t >= window_start]
        if len(self.pages_in_window) >= C.max_pages_per_2h:
            return False
        stats = self.fp_counter.get(action, {"accepted":0, "rejected":0})
        total = stats["accepted"] + stats["rejected"]
        if total >= 20 and (stats["rejected"]/max(total,1)) > C.fp_budget_per_h.get(action, 1.0):
            return False
        # emit
        self.last_by_role[role] = now_ts
        self.last_action_ts[key] = now_ts
        self.pages_in_window.append(now_ts)
        return True
    def feedback(self, action: str, accepted: bool):
        stats = self.fp_counter.setdefault(action, {"accepted":0, "rejected":0})
        if accepted: stats["accepted"] += 1
        else:        stats["rejected"] += 1


## Layer 1 — Skill Library (small, legible skills)

In [ ]:

@dataclass
class WorkflowState:
    encounter_id: Optional[str] = None
    patient_id: Optional[str] = None
    pending_orders: set = field(default_factory=set)
    completed_studies: set = field(default_factory=set)
    active_consults: set = field(default_factory=set)
    last_vitals_ts: Optional[pd.Timestamp] = None
    chest_pain: bool = False
    trauma: bool = False
    # context
    backlog_ct: int = 0
    backlog_lab: int = 0
    backlog_ecg: int = 0
    hour: int = 12
    role: str = "nurse"

def skill_need_ecg(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if state.chest_pain and ("ORDER_ECG" not in state.pending_orders) and ("ORDER_ECG" not in state.completed_studies):
        return {"action":"ORDER_ECG", "reason":"Chest pain without ECG", "urgency":"high"}
    return None

def skill_abnormal_ecg_no_consult(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_ECG" in state.completed_studies) and ("ECG_ABNORMAL" in state.completed_studies) and ("CARDIOLOGY" not in state.active_consults):
        return {"action":"PAGE_CARDIOLOGY", "reason":"Abnormal ECG without consult", "urgency":"high"}
    return None

def skill_ct_delayed(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_CT" in state.pending_orders) and ("CT_RESULT" not in state.completed_studies):
        return {"action":"FOLLOW_UP_IMAGING", "reason":"CT pending > 60m", "urgency":"medium"}
    return None

def skill_pending_labs_deteriorating(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if (("LAB_TROPONIN" in state.pending_orders) or ("LAB_PANEL" in state.pending_orders)) and ("Deteriorating" in state.completed_studies):
        return {"action":"EXPEDITE_LABS", "reason":"Pending labs + deterioration", "urgency":"high"}
    return None

SKILLS = [
    skill_need_ecg,
    skill_abnormal_ecg_no_consult,
    skill_ct_delayed,
    skill_pending_labs_deteriorating,
]

def generate_candidates(state: WorkflowState) -> List[Dict[str,Any]]:
    out = []
    for s in SKILLS:
        r = s(state)
        if r: out.append(r)
    return out[:5]


## Layer 2 — Critics (acceptance / benefit / burden)

In [ ]:

class TinyCritics:
    def __init__(self):
        # acceptance classifier (LogReg + isotonic calibration)
        base = Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("scale", StandardScaler(with_mean=False)),
            ("clf", LogisticRegression(max_iter=300, C=0.5, class_weight="balanced", random_state=4242)),
        ])
        self.accept = CalibratedClassifierCV(base, method="isotonic", cv=3)
        # benefit regressor: clicks saved proxy (we'll emulate with simple rule for now)
        self.benefit_bias = {"ORDER_ECG": 3.0, "PAGE_CARDIOLOGY": 2.0, "FOLLOW_UP_IMAGING": 1.5, "EXPEDITE_LABS": 2.0, "ORDER_CAP_BGA": 2.0, "FOLLOW_UP_CONSULT": 1.0}
        # burden proxy
        self.burden_bias = {"ORDER_ECG": 0.2, "PAGE_CARDIOLOGY": 0.5, "FOLLOW_UP_IMAGING": 0.3, "EXPEDITE_LABS": 0.2, "ORDER_CAP_BGA": 0.2, "FOLLOW_UP_CONSULT": 0.3}
        self.is_fit = False

    def _featurize(self, X: List[Dict[str,Any]]) -> pd.DataFrame:
        # Simple, robust features from state+action
        rows = []
        for x in X:
            st: WorkflowState = x["state"]
            a: str = x["action"]["action"]
            rows.append({
                "hour": st.hour,
                "role_is_nurse": 1 if st.role=="nurse" else 0,
                "role_is_resident": 1 if st.role=="resident" else 0,
                "backlog_ct": st.backlog_ct,
                "backlog_lab": st.backlog_lab,
                "backlog_ecg": st.backlog_ecg,
                "chest_pain": int(st.chest_pain),
                "trauma": int(st.trauma),
                "act_ORDER_ECG": 1 if a=="ORDER_ECG" else 0,
                "act_PAGE_CARDIOLOGY": 1 if a=="PAGE_CARDIOLOGY" else 0,
                "act_FOLLOW_UP_IMAGING": 1 if a=="FOLLOW_UP_IMAGING" else 0,
                "act_EXPEDITE_LABS": 1 if a=="EXPEDITE_LABS" else 0,
            })
        return pd.DataFrame(rows)

    def fit(self, logs: pd.DataFrame):
        """logs columns: [hour, role, backlog_ct, backlog_lab, backlog_ecg, chest_pain, trauma, action, accepted]"""
        # build features from logs
        X_dicts = []
        for _, r in logs.iterrows():
            st = WorkflowState(hour=int(r["hour"]), role=r["role"],
                               backlog_ct=int(r["backlog_ct"]), backlog_lab=int(r["backlog_lab"]), backlog_ecg=int(r["backlog_ecg"]),
                               chest_pain=bool(r["chest_pain"]), trauma=bool(r["trauma"]))
            X_dicts.append({"state": st, "action": {"action": r["action"]}})
        X = self._featurize(X_dicts)
        y = logs["accepted"].astype(int).values
        self.accept.fit(X, y)
        self.is_fit = True

    def score(self, state: WorkflowState, actions: List[Dict[str,Any]]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        # acceptance prob, benefit, burden
        X = self._featurize([{"state": state, "action": a} for a in actions])
        if self.is_fit:
            p = self.accept.predict_proba(X)[:,1]
        else:
            # prior: moderate acceptance, adjusted by urgency
            p = np.array([0.6 if a.get("urgency")=="high" else 0.5 for a in actions], dtype=float)
        benefit = np.array([self.benefit_bias.get(a["action"], 1.5) for a in [x["action"] for x in [{"action":a} for a in actions]]], dtype=float)
        burden = np.array([self.burden_bias.get(a["action"], 0.3) for a in [x["action"] for x in [{"action":a} for a in actions]]], dtype=float)
        return p, benefit, burden


## Layer 3 — Meta-controller (bounded agency)

In [ ]:

@dataclass
class ControllerConfig:
    w_accept: float = 1.0
    w_benefit: float = 0.3
    w_burden: float = 0.5
    explore_eps: float = 0.05  # Thompson mixing factor

class MetaController:
    def __init__(self, critics: TinyCritics, cfg: ControllerConfig):
        self.critics = critics
        self.cfg = cfg
        self.beta_priors = {a:(1,1) for a in ["ORDER_ECG","PAGE_CARDIOLOGY","FOLLOW_UP_IMAGING","EXPEDITE_LABS","ORDER_CAP_BGA","FOLLOW_UP_CONSULT","FOLLOW_UP_CATH_ACTIVATION","FOLLOW_UP_ENDO_ACTIVATION"]}

    def select(self, state: WorkflowState, candidates: List[Dict[str,Any]]) -> Optional[Dict[str,Any]]:
        if not candidates:
            return None
        p, ben, bur = self.critics.score(state, candidates)
        # Thompson on acceptance
        th = np.array([np.random.beta(*self.beta_priors[a["action"]]) for a in candidates])
        utility = self.cfg.w_accept*p + self.cfg.w_benefit*ben - self.cfg.w_burden*bur + self.cfg.explore_eps*th
        i = int(np.argmax(utility))
        return candidates[i]

    def feedback(self, action: str, accepted: bool):
        a,b = self.beta_priors.get(action, (1,1))
        self.beta_priors[action] = (a+1, b) if accepted else (a, b+1)


## Layer 4 — Memory (casebook / retrieval)

In [ ]:

class Casebook:
    def __init__(self, k:int=20):
        self.k = k
        self.df = pd.DataFrame(columns=["hour","role","backlog_ct","backlog_lab","backlog_ecg","chest_pain","trauma","action","accepted"])
        self.nn = None
        self._fit_needed = True
    def add(self, hour, role, backlog_ct, backlog_lab, backlog_ecg, chest_pain, trauma, action, accepted):
        self.df.loc[len(self.df)] = [hour, role, backlog_ct, backlog_lab, backlog_ecg, int(chest_pain), int(trauma), action, int(accepted)]
        self._fit_needed = True
    def build(self):
        if self.df.empty:
            return
        X = self.df[["hour","backlog_ct","backlog_lab","backlog_ecg","chest_pain","trauma"]].values
        self.nn = NearestNeighbors(n_neighbors=min(self.k, len(X)), metric="euclidean").fit(X)
        self._fit_needed = False
    def retrieve(self, state: WorkflowState):
        if self._fit_needed: self.build()
        if self.nn is None or self.df.empty:
            return []
        q = np.array([[state.hour, state.backlog_ct, state.backlog_lab, state.backlog_ecg, int(state.chest_pain), int(state.trauma)]])
        idx = self.nn.kneighbors(q, return_distance=False)[0]
        return self.df.iloc[idx].to_dict(orient="records")


## Replay loop (works with simulated or real event logs)

In [ ]:

def update_state_from_event(state: WorkflowState, row: pd.Series):
    et = row["event_type"]
    if et.startswith("ORDER_"):
        state.pending_orders.add(et)
    if et.endswith("_RESULT"):
        state.completed_studies.add(et.replace("_RESULT",""))
    if et in ("ECG_ABNORMAL","Deteriorating"):
        state.completed_studies.add(et)
    if et.startswith("PAGE_"):
        state.active_consults.add(et.split("_",1)[1])
    if et == "VITALS":
        state.last_vitals_ts = row["ts"]
    if et == "CHEST_PAIN":
        state.chest_pain = True
    if et == "TRAUMA":
        state.trauma = True

def replay(events: pd.DataFrame, critics: TinyCritics, controller: MetaController, constraints: Constraints, seed=13):
    rng = np.random.default_rng(seed)
    Cstate = ConstraintState()
    casebook = Casebook(k=50)

    emissions = []
    counters = {"suggestions":0, "accepted":0, "rejected":0, "clicks_saved":0, "seconds_saved":0.0}

    for enc_id, grp in events.groupby("encounter_id"):
        st = WorkflowState(encounter_id=enc_id)
        for _, row in grp.iterrows():
            # load minimal context
            st.role = row.get("role","nurse")
            st.hour = int(pd.to_datetime(row["ts"]).hour)
            # backlog proxies (optional columns)
            st.backlog_ct = int(row.get("backlog_ct", 0))
            st.backlog_lab = int(row.get("backlog_lab", 0))
            st.backlog_ecg = int(row.get("backlog_ecg", 0))

            update_state_from_event(st, row)

            if row["event_type"] in ("TRIAGE","VITALS","ECG_RESULT","CT_RESULT","LAB_RESULT","NURSE_ROUND"):
                cands = generate_candidates(st)
                a = controller.select(st, cands)
                if not a: 
                    continue
                allowed = Cstate.allow(pd.to_datetime(row["ts"]).timestamp(), st.role, st.encounter_id, a["action"], constraints)
                if not allowed:
                    continue

                counters["suggestions"] += 1
                # acceptance: if urgent/abnormal reason then more likely
                p_acc, _, _ = critics.score(st, [a])
                accepted = (rng.random() < float(p_acc[0]))
                if accepted: counters["accepted"] += 1
                else: counters["rejected"] += 1
                Cstate.feedback(a["action"], accepted)
                controller.feedback(a["action"], accepted)
                casebook.add(st.hour, st.role, st.backlog_ct, st.backlog_lab, st.backlog_ecg, st.chest_pain, st.trauma, a["action"], accepted)


                # per-action benefit accounting
                benefit_clicks = {"ORDER_ECG":3,"ORDER_CAP_BGA":2,"ORDER_LABS":3,
                                  "FOLLOW_UP_IMAGING":1,"FOLLOW_UP_CONSULT":1,
                                  "PAGE_CARDIOLOGY":1,"EXPEDITE_LABS":1}
                benefit_seconds = {"ORDER_ECG":20.0,"ORDER_CAP_BGA":15.0,"ORDER_LABS":25.0,
                                   "FOLLOW_UP_IMAGING":10.0,"FOLLOW_UP_CONSULT":12.0,
                                   "PAGE_CARDIOLOGY":8.0,"EXPEDITE_LABS":10.0}
                if accepted:
                    counters["clicks_saved"] += int(benefit_clicks.get(a["action"], 0))
                    counters["seconds_saved"] += float(benefit_seconds.get(a["action"], 0.0))

                emissions.append({"ts": str(row["ts"]), "encounter_id": st.encounter_id, "action": a["action"], "accepted": accepted, "reason": a.get("reason")})

    # Fit critics on casebook (acceptance)
    if len(casebook.df) >= 30:
        critics.fit(casebook.df.rename(columns={"accepted":"accepted"}))
    # Save artifacts
    with open(ART / "emissions.json","w") as f: json.dump(emissions, f, indent=2)
    with open(ART / "metrics.json","w") as f: json.dump({
        **counters,
        "acceptance_rate": counters["accepted"]/max(counters["suggestions"],1),
        "alert_fatigue_proxy": counters["rejected"]/max(counters["suggestions"],1)
    }, f, indent=2)
    return counters, casebook.df


## Simulation fallback (if you don’t have real logs yet)

In [ ]:

def simulate_events(hours=12, base_rate=3.0, seed=17):
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2025-01-01T00:00:00Z")
    rows = []
    enc = 0
    for h in range(hours):
        n = rng.poisson(max(0.1, base_rate + 2.0*np.sin(h/24*2*np.pi)))
        for i in range(n):
            enc += 1
            pid = f"P{enc:05d}"
            eid = f"E{enc:05d}"
            t0 = start + pd.Timedelta(hours=h, minutes=rng.uniform(0,60))
            rows.append([t0, pid, eid, "nurse", "ED", "TRIAGE", "{}"])
            if rng.random() < 0.4:
                rows.append([t0 + pd.Timedelta(minutes=5), pid, eid, "nurse", "ED", "VITALS", "{}"])
            cp = rng.random() < 0.25
            if cp:
                rows.append([t0 + pd.Timedelta(minutes=6), pid, eid, "ED_MD", "ED", "CHEST_PAIN", "{}"])
                rows.append([t0 + pd.Timedelta(minutes=8), pid, eid, "ED_MD", "ED", "ORDER_ECG", "{}"])
                rows.append([t0 + pd.Timedelta(minutes=18), pid, eid, "tech", "ED", "ECG_RESULT", "{}"])
                if rng.random() < 0.2:
                    rows.append([t0 + pd.Timedelta(minutes=18), pid, eid, "tech", "ED", "ECG_ABNORMAL", "{}"])
            if rng.random() < 0.15:
                rows.append([t0 + pd.Timedelta(minutes=12), pid, eid, "ED_MD", "ED", "ORDER_CT", "{}"])
                rows.append([t0 + pd.Timedelta(minutes=50), pid, eid, "tech", "CT", "CT_RESULT", "{}"])
            if rng.random() < 0.2:
                rows.append([t0 + pd.Timedelta(minutes=20), pid, eid, "lab", "LAB", "LAB_TROPONIN", "{}"])
                rows.append([t0 + pd.Timedelta(minutes=65), pid, eid, "lab", "LAB", "LAB_RESULT", "{}"])
            # periodic rounds
            for k in range(3):
                rows.append([t0 + pd.Timedelta(minutes=30*(k+1)), pid, eid, "nurse", "ED", "NURSE_ROUND", "{}"])
    df = pd.DataFrame(rows, columns=["ts","patient_id","encounter_id","role","location","event_type","event_attrs"])
    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    return df.sort_values("ts").reset_index(drop=True)

events_demo = simulate_events(hours=24, base_rate=4.0)
events_demo.head(3)


## Run replay with the new architecture

In [ ]:

critics = TinyCritics()
controller = MetaController(critics, ControllerConfig())
constraints = Constraints()

counters, case_df = replay(events_demo, critics, controller, constraints)
print(json.dumps({"metrics":{
    **counters,
    "acceptance_rate": counters["accepted"]/max(counters["suggestions"],1),
    "alert_fatigue_proxy": counters["rejected"]/max(counters["suggestions"],1)
}}, indent=2))

print("Artifacts:", [p.name for p in ART.iterdir()])



## Data Quality Checks (Kaggle-friendly)
**Hardened ID detection, overlap across splits, and prevalence.**  
If train/val/test CSVs are present in your Kaggle dataset, this cell will find them and run checks. Otherwise it will no-op.


In [ ]:

import os, re, json, pandas as pd, numpy as np

def _find_files(root="/kaggle/input"):
    hits = {"train": None, "val": None, "test": None}
    if not os.path.exists(root):
        return hits
    for dirpath, dirnames, filenames in os.walk(root):
        names = {f.lower(): os.path.join(dirpath, f) for f in filenames}
        for key, pat in [("train","train_de_full.csv"), ("val","val_de_full.csv"), ("test","test_de_full.csv")]:
            if hits[key] is None:
                # exact or approximate match
                for fname, full in names.items():
                    if pat == fname or fname.endswith(pat) or ("_"+key in fname and "de" in fname and "full" in fname and fname.endswith(".csv")):
                        hits[key] = full; break
    return hits

def id_cols_authoritative_de(df):
    base = [
        r'^patient_id$', r'^patient_key$', r'^mrn$', r'^medical_record_number$',
        r'^person_id$', r'^subject_id$', r'^individual_id$',
        r'^encounter_id$', r'^encounter_key$', r'^visit_id$', r'^visit_key$',
        r'^episode_id$', r'^episode_key$', r'^admission_id$', r'^stay_id$',
        r'^case_id$', r'^case_number$', r'^account_id$', r'^account_number$',
        r'^chart_id$', r'^chart_number$', r'^order_id$', r'^procedure_id$',
        r'^provider_id$', r'^attending_id$', r'^nurse_id$', r'^bed_id$',
        r'^room_id$', r'^unit_id$', r'^department_id$',
        r'^id$', r'_id$', r'^.*_key$', r'^key_.*$',
        r'identifier', r'reference', r'external_id'
    ]
    de = [
        r'^fall[-_]?id$', r'^fallid$', r'^fallnummer$', r'^fall_nummer$', r'^fallnr$',
        r'^aufnahme[-_]?id$', r'^aufenthalt[-_]?id$', r'^aufenthaltsnummer$',
        r'^patient[-_]?id$', r'^patientennummer$', r'^versichertennummer$', r'^versnr$', r'^kvnr$',
        r'^behandlungsfall[-_]?id$', r'^fall[-_]?key$'
    ]
    pats = base + de
    out = []
    for col in df.columns:
        s = col.lower().strip()
        if any(re.match(p, s) for p in pats):
            out.append(col)
    return sorted(set(out))

def validate_id_column(df, col):
    s = df[col].dropna()
    if s.empty: return False
    sample = s.sample(min(len(s), 1000), random_state=13)
    if s.dtype == 'O' or pd.api.types.is_string_dtype(s):
        has_letters = sample.astype(str).str.contains(r'[A-Za-z]').any()
        has_complex = sample.astype(str).str.len().var() > 0
        many_uniques = s.nunique() > 0.5 * len(s)
        return bool(has_letters or has_complex or many_uniques)
    diffs = sample.sort_values().diff().dropna()
    is_seq = diffs.nunique()==1 and float(diffs.iloc[0])==1.0
    many_uniques = s.nunique() > 0.5 * len(s)
    return bool((not is_seq) or many_uniques)

def safe_overlap(col, a, b):
    if col not in a.columns or col not in b.columns: return "N/A"
    return int(len(set(a[col].dropna()) & set(b[col].dropna())))

hits = _find_files()
print("Detected CSVs:", hits)
if all(hits.values()):
    train = pd.read_csv(hits["train"])
    val   = pd.read_csv(hits["val"])
    test  = pd.read_csv(hits["test"])

    cands = sorted(set(id_cols_authoritative_de(train)+id_cols_authoritative_de(val)+id_cols_authoritative_de(test)))
    concat = pd.concat([d.reindex(columns=cands, fill_value=np.nan) for d in [train,val,test]], ignore_index=True) if cands else pd.DataFrame()
    validated = []
    for c in cands:
        try:
            if validate_id_column(concat, c):
                validated.append(c)
        except Exception:
            validated.append(c)

    overlap = []
    for c in validated:
        overlap.append({
            "id_col": c,
            "train∩val":  safe_overlap(c, train, val),
            "train∩test": safe_overlap(c, train, test),
            "val∩test":   safe_overlap(c, val, test)
        })
    overlap_df = pd.DataFrame(overlap).sort_values("id_col")
    display(overlap_df)

    # Prevalence
    label_col = next((c for c in train.columns if c.lower() in {"label","target","y","kap_veraltet"}), None)
    if label_col is None:
        for c in train.columns:
            if pd.api.types.is_numeric_dtype(train[c]) and set(pd.unique(train[c].dropna())) <= {0,1}:
                label_col = c; break
    if label_col:
        prev = pd.DataFrame([
            {"split":"train","pos_rate":float(train[label_col].mean()),"n":len(train)},
            {"split":"val",  "pos_rate":float(val[label_col].mean()),  "n":len(val)},
            {"split":"test", "pos_rate":float(test[label_col].mean()), "n":len(test)},
        ])
        display(prev)
else:
    print("No train/val/test CSVs found under /kaggle/input — skipping checks.")



## Clinical Adjustments (EMS handoff, universal ECG + venous BGA, SpO₂, conditional capillary BGA, consults)
This cell **overrides** the earlier `WorkflowState`, `update_state_from_event`, `SKILLS`, and `simulate_events(...)` to reflect your domain:
- **Everyone**: `EMS_HANDOFF`, `TRIAGE`, `HISTORY`, `VITALS` (with `SpO2`), `ORDER_ECG`, `ORDER_LABS` (includes **venous BGA**).
- **Breathing problem**: also `ORDER_CAP_BGA` (capillary BGA).
- **Consults**: sampled from a set of ~10 consult types, biased by presentation.


In [ ]:

# --- Override: richer WorkflowState & skills ---
from dataclasses import dataclass, field
from typing import Optional, Any, Dict, List
import pandas as pd, numpy as np, json, datetime as dt

@dataclass
class WorkflowState:
    encounter_id: Optional[str] = None
    patient_id: Optional[str] = None
    pending_orders: set = field(default_factory=set)
    completed_studies: set = field(default_factory=set)
    active_consults: set = field(default_factory=set)
    last_vitals_ts: Optional[pd.Timestamp] = None
    chest_pain: bool = False
    breathing_problem: bool = False
    trauma: bool = False
    spo2: Optional[float] = None
    # context
    backlog_ct: int = 0
    backlog_lab: int = 0
    backlog_ecg: int = 0
    hour: int = 12
    role: str = "nurse"

def update_state_from_event(state: WorkflowState, row: pd.Series):
    et = row["event_type"]
    attrs = row.get("event_attrs", "{}")
    if isinstance(attrs, str):
        try: attrs = json.loads(attrs)
        except: attrs = {}
    if et == "VITALS":
        state.last_vitals_ts = row["ts"]
        if "spo2" in attrs:
            try: state.spo2 = float(attrs["spo2"])
            except: pass
    if et == "CHEST_PAIN":
        state.chest_pain = True
    if et == "BREATHING":
        state.breathing_problem = True
    if et == "TRAUMA":
        state.trauma = True
    if et.startswith("ORDER_"):
        state.pending_orders.add(et)
    if et.endswith("_RESULT"):
        state.completed_studies.add(et.replace("_RESULT",""))
    if et in ("ECG_ABNORMAL","Deteriorating"):
        state.completed_studies.add(et)
    if et.startswith("PAGE_"):
        state.active_consults.add(et.split("_",1)[1])
    if et.startswith("REQUEST_CONSULT_"):
        state.pending_orders.add(et)

def skill_need_cap_bga(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if state.breathing_problem and ("ORDER_CAP_BGA" not in state.pending_orders) and ("ORDER_CAP_BGA" not in state.completed_studies):
        # if hypoxic vitals, increase urgency
        urgency = "high" if (state.spo2 is not None and state.spo2 < 92) else "medium"
        return {"action":"ORDER_CAP_BGA", "reason":"Breathing problem without capillary BGA", "urgency": urgency}
    return None

def skill_follow_up_consult(state: WorkflowState) -> Optional[Dict[str,Any]]:
    # If any consult requested but not acknowledged/resulted, nudge follow-up
    pend = [x for x in state.pending_orders if isinstance(x, str) and x.startswith("REQUEST_CONSULT_")]
    if pend:
        return {"action":"FOLLOW_UP_CONSULT", "reason":"Consult requested pending acknowledgement", "urgency":"medium"}
    return None

# Keep prior skills (need_ecg will usually be no-op since everyone orders ECG in sim)
def skill_need_ecg(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_ECG" not in state.pending_orders) and ("ORDER_ECG" not in state.completed_studies):
        return {"action":"ORDER_ECG", "reason":"Baseline ECG missing", "urgency":"high"}
    return None

def skill_abnormal_ecg_no_consult(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_ECG" in state.completed_studies) and ("ECG_ABNORMAL" in state.completed_studies) and ("CARDIOLOGY" not in state.active_consults):
        return {"action":"PAGE_CARDIOLOGY", "reason":"Abnormal ECG without consult", "urgency":"high"}
    return None

def skill_ct_delayed(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_CT" in state.pending_orders) and ("CT_RESULT" not in state.completed_studies):
        return {"action":"FOLLOW_UP_IMAGING", "reason":"CT pending > 60m", "urgency":"medium"}
    return None

def skill_pending_labs_deteriorating(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if (("ORDER_LABS" in state.pending_orders) or ("LAB_PANEL" in state.pending_orders)) and ("Deteriorating" in state.completed_studies):
        return {"action":"EXPEDITE_LABS", "reason":"Pending labs + deterioration", "urgency":"high"}
    return None

SKILLS = [
    skill_need_cap_bga,
    skill_follow_up_consult,
    skill_abnormal_ecg_no_consult,
    skill_ct_delayed,
    skill_pending_labs_deteriorating,
    skill_need_ecg,
]

# (Removed: class-level TinyCritics bias update; defaults set in __init__)


In [ ]:

# --- Override: clinically tuned simulator ---
def simulate_events(hours=24, base_rate=4.0, seed=23):
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2025-01-01T00:00:00Z", tz="UTC")
    rows = []
    enc = 0

    CONSULT_TYPES = ["CARDIOLOGY","PULMONOLOGY","NEUROLOGY","GENERAL_SURGERY","ORTHOPEDICS",
                     "NEPHROLOGY","GASTRO","ENT","ICU","RADIOLOGY"]

    for h in range(hours):
        n = rng.poisson(max(0.5, base_rate + 2.0*np.sin(h/24*2*np.pi)))
        for i in range(n):
            enc += 1
            pid = f"P{enc:06d}"
            eid = f"E{enc:06d}"
            t0 = start + pd.Timedelta(hours=h, minutes=rng.uniform(0,60))

            # EMS handoff
            rows.append([t0, pid, eid, "nurse", "ED", "EMS_HANDOFF", "{}"])
            # TRIAGE
            t_triage = t0 + pd.Timedelta(minutes=rng.uniform(1,6))
            rows.append([t_triage, pid, eid, "nurse", "ED", "TRIAGE", "{}"])
            # HISTORY
            t_hist = t_triage + pd.Timedelta(minutes=rng.uniform(1,4))
            rows.append([t_hist, pid, eid, "ED_MD", "ED", "HISTORY", "{}"])

            # Presentation flags
            chest_pain = rng.random() < 0.22
            breathing  = rng.random() < 0.18
            trauma     = rng.random() < 0.10

            if chest_pain:
                rows.append([t_hist, pid, eid, "ED_MD", "ED", "CHEST_PAIN", "{}"])
            if breathing:
                rows.append([t_hist + pd.Timedelta(minutes=0.5), pid, eid, "ED_MD", "ED", "BREATHING", "{}"])
            if trauma:
                rows.append([t_hist + pd.Timedelta(minutes=1), pid, eid, "ED_MD", "ED", "TRAUMA", "{}"])

            # VITALS (include SpO2)
            spo2_mu = 97 if not breathing else 90
            spo2_sd = 1.5 if not breathing else 3.0
            spo2 = float(np.clip(rng.normal(spo2_mu, spo2_sd), 75, 100))
            rows.append([t_hist + pd.Timedelta(minutes=2), pid, eid, "nurse", "ED",
                         "VITALS", json.dumps({"spo2": round(spo2,1)})])

            # Universal ECG order + result
            t_order_ecg = t_hist + pd.Timedelta(minutes=rng.uniform(2,8))
            rows.append([t_order_ecg, pid, eid, "ED_MD", "ED", "ORDER_ECG", "{}"])
            t_ecg_res = t_order_ecg + pd.Timedelta(minutes=rng.normal(10,4))
            rows.append([t_ecg_res, pid, eid, "tech", "ED", "ECG_RESULT", "{}"])
            # abnormalities biased by chest pain
            p_abn = 0.18 if chest_pain else 0.05
            if rng.random() < p_abn:
                rows.append([t_ecg_res, pid, eid, "tech", "ED", "ECG_ABNORMAL", "{}"])

            # Universal labs incl venous BGA
            t_labs = t_hist + pd.Timedelta(minutes=rng.uniform(3,10))
            rows.append([t_labs, pid, eid, "ED_MD", "ED", "ORDER_LABS", "{}"])
            t_lab_res = t_labs + pd.Timedelta(minutes=max(10, rng.normal(45, 15)))
            rows.append([t_lab_res, pid, eid, "lab", "LAB", "LAB_RESULT", "{}"])

            # Capillary BGA for breathing problems
            if breathing:
                t_cap = t_hist + pd.Timedelta(minutes=rng.uniform(4,12))
                rows.append([t_cap, pid, eid, "ED_MD", "ED", "ORDER_CAP_BGA", "{}"])
                t_cap_res = t_cap + pd.Timedelta(minutes=max(5, rng.normal(20,8)))
                rows.append([t_cap_res, pid, eid, "lab", "LAB", "CAP_BGA_RESULT", "{}"])

            # Imaging occasionally
            if rng.random() < (0.30 if trauma else 0.12):
                t_ct = t_hist + pd.Timedelta(minutes=rng.uniform(8,22))
                rows.append([t_ct, pid, eid, "ED_MD", "ED", "ORDER_CT", "{}"])
                t_ct_res = t_ct + pd.Timedelta(minutes=max(20, rng.normal(35, 12)))
                rows.append([t_ct_res, pid, eid, "tech", "CT", "CT_RESULT", "{}"])

            # Consult ordering (biased)
            consult = None
            if chest_pain and rng.random() < 0.50:
                consult = "CARDIOLOGY"
            elif breathing and rng.random() < 0.45:
                consult = "PULMONOLOGY"
            elif trauma and rng.random() < 0.40:
                consult = rng.choice(["GENERAL_SURGERY","ORTHOPEDICS"])
            elif rng.random() < 0.20:
                consult = rng.choice(CONSULT_TYPES)
            if consult:
                t_req = t_hist + pd.Timedelta(minutes=rng.uniform(5,25))
                rows.append([t_req, pid, eid, "ED_MD", "ED", f"REQUEST_CONSULT_{consult}", "{}"])
                # ack may be delayed
                if rng.random() < 0.7:
                    t_ack = t_req + pd.Timedelta(minutes=rng.uniform(10,45))
                    rows.append([t_ack, pid, eid, "specialist", consult, "CONSULT_ACK", "{}"])

            # Rounds
            for k in range(1,5):
                rows.append([t_hist + pd.Timedelta(minutes=15*k), pid, eid, "nurse", "ED", "NURSE_ROUND", "{}"])

    df = pd.DataFrame(rows, columns=["ts","patient_id","encounter_id","role","location","event_type","event_attrs"])
    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    return df.sort_values("ts").reset_index(drop=True)

# Generate a larger demo with the clinical assumptions
events_demo = simulate_events(hours=24, base_rate=4.0, seed=42)
print("Simulated events:", len(events_demo))
events_demo.head(5)



## Shift-aware interventions (Cath lab & Endoscopy)
Night (22:00–06:59) requires **~30 minutes** from call to intervention; daytime targets **~15 minutes**.
This cell overrides state, simulator, and skills accordingly.


In [ ]:

# --- Override WorkflowState & updater to track times ---
from dataclasses import dataclass, field
from typing import Optional, Any, Dict, List
import pandas as pd, numpy as np, json

@dataclass
class WorkflowState:
    encounter_id: Optional[str] = None
    patient_id: Optional[str] = None
    pending_orders: set = field(default_factory=set)
    completed_studies: set = field(default_factory=set)
    active_consults: set = field(default_factory=set)
    last_vitals_ts: Optional[pd.Timestamp] = None
    chest_pain: bool = False
    breathing_problem: bool = False
    trauma: bool = False
    spo2: Optional[float] = None
    # context
    backlog_ct: int = 0
    backlog_lab: int = 0
    backlog_ecg: int = 0
    hour: int = 12
    role: str = "nurse"
    # timing
    now_ts: Optional[pd.Timestamp] = None
    event_times: Dict[str, pd.Timestamp] = field(default_factory=dict)

def update_state_from_event(state: WorkflowState, row: pd.Series):
    et = row["event_type"]
    ts = pd.to_datetime(row["ts"], utc=True)
    state.now_ts = ts
    attrs = row.get("event_attrs", "{}")
    if isinstance(attrs, str):
        try: attrs = json.loads(attrs)
        except: attrs = {}
    # vitals / SpO2
    if et == "VITALS":
        state.last_vitals_ts = ts
        if "spo2" in attrs:
            try: state.spo2 = float(attrs["spo2"])
            except: pass
    # flags
    if et == "CHEST_PAIN": state.chest_pain = True
    if et == "BREATHING": state.breathing_problem = True
    if et == "TRAUMA": state.trauma = True
    # orders / results / consults
    if et.startswith("ORDER_") or et.startswith("REQUEST_CONSULT_"):
        state.pending_orders.add(et)
    if et.endswith("_RESULT") or et.startswith("INTERVENTION_START_"):
        # mark base token as completed
        base = et.replace("_RESULT","")
        state.completed_studies.add(base)
    if et.startswith("PAGE_"):
        state.active_consults.add(et.split("_",1)[1])
    # remember timestamps
    state.event_times[et] = ts


In [ ]:

# --- Override simulator with shift-aware intervention delays for cath & endo ---
def is_night(ts: pd.Timestamp) -> bool:
    h = int(pd.to_datetime(ts).hour)
    return (h >= 22 or h < 7)

def simulate_events(hours=24, base_rate=4.0, seed=99):
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2025-01-01T00:00:00Z", tz="UTC")
    rows = []
    enc = 0

    CONSULT_TYPES = ["CARDIOLOGY","PULMONOLOGY","NEUROLOGY","GENERAL_SURGERY","ORTHOPEDICS",
                     "NEPHROLOGY","GASTRO","ENT","ICU","RADIOLOGY"]

    for h in range(hours):
        n = rng.poisson(max(0.5, base_rate + 2.0*np.sin(h/24*2*np.pi)))
        for i in range(n):
            enc += 1
            pid = f"P{enc:06d}"
            eid = f"E{enc:06d}"
            t0 = start + pd.Timedelta(hours=h, minutes=rng.uniform(0,60))

            # Core flow
            rows.append([t0, pid, eid, "nurse", "ED", "EMS_HANDOFF", "{}"])
            t_triage = t0 + pd.Timedelta(minutes=rng.uniform(1,6))
            rows.append([t_triage, pid, eid, "nurse", "ED", "TRIAGE", "{}"])
            t_hist = t_triage + pd.Timedelta(minutes=rng.uniform(1,4))
            rows.append([t_hist, pid, eid, "ED_MD", "ED", "HISTORY", "{}"])

            chest_pain = rng.random() < 0.22
            breathing  = rng.random() < 0.18
            trauma     = rng.random() < 0.10
            if chest_pain: rows.append([t_hist, pid, eid, "ED_MD", "ED", "CHEST_PAIN", "{}"])
            if breathing:  rows.append([t_hist + pd.Timedelta(minutes=0.5), pid, eid, "ED_MD", "ED", "BREATHING", "{}"])
            if trauma:     rows.append([t_hist + pd.Timedelta(minutes=1), pid, eid, "ED_MD", "ED", "TRAUMA", "{}"])

            spo2_mu = 97 if not breathing else 90
            spo2_sd = 1.5 if not breathing else 3.0
            spo2 = float(np.clip(rng.normal(spo2_mu, spo2_sd), 75, 100))
            rows.append([t_hist + pd.Timedelta(minutes=2), pid, eid, "nurse", "ED", "VITALS", json.dumps({"spo2": round(spo2,1)})])

            # Universal orders
            t_order_ecg = t_hist + pd.Timedelta(minutes=rng.uniform(2,8))
            rows.append([t_order_ecg, pid, eid, "ED_MD", "ED", "ORDER_ECG", "{}"])
            t_ecg_res = t_order_ecg + pd.Timedelta(minutes=max(5, rng.normal(10,4)))
            rows.append([t_ecg_res, pid, eid, "tech", "ED", "ECG_RESULT", "{}"])
            if rng.random() < (0.18 if chest_pain else 0.05):
                rows.append([t_ecg_res, pid, eid, "tech", "ED", "ECG_ABNORMAL", "{}"])

            t_labs = t_hist + pd.Timedelta(minutes=rng.uniform(3,10))
            rows.append([t_labs, pid, eid, "ED_MD", "ED", "ORDER_LABS", "{}"])
            t_lab_res = t_labs + pd.Timedelta(minutes=max(10, rng.normal(45, 15)))
            rows.append([t_lab_res, pid, eid, "lab", "LAB", "LAB_RESULT", "{}"])

            if breathing:
                t_cap = t_hist + pd.Timedelta(minutes=rng.uniform(4,12))
                rows.append([t_cap, pid, eid, "ED_MD", "ED", "ORDER_CAP_BGA", "{}"])
                t_cap_res = t_cap + pd.Timedelta(minutes=max(5, rng.normal(20,8)))
                rows.append([t_cap_res, pid, eid, "lab", "LAB", "CAP_BGA_RESULT", "{}"])

            # Imaging occasionally
            if rng.random() < (0.30 if trauma else 0.12):
                t_ct = t_hist + pd.Timedelta(minutes=rng.uniform(8,22))
                rows.append([t_ct, pid, eid, "ED_MD", "ED", "ORDER_CT", "{}"])
                t_ct_res = t_ct + pd.Timedelta(minutes=max(20, rng.normal(35, 12)))
                rows.append([t_ct_res, pid, eid, "tech", "CT", "CT_RESULT", "{}"])

            # Consults (biased)
            consult = None
            if chest_pain and rng.random() < 0.50:
                consult = "CARDIOLOGY"
            elif breathing and rng.random() < 0.45:
                consult = "PULMONOLOGY"
            elif trauma and rng.random() < 0.40:
                consult = rng.choice(["GENERAL_SURGERY","ORTHOPEDICS"])
            elif rng.random() < 0.20:
                consult = rng.choice(CONSULT_TYPES)

            if consult:
                t_req = t_hist + pd.Timedelta(minutes=rng.uniform(5,25))
                rows.append([t_req, pid, eid, "ED_MD", "ED", f"REQUEST_CONSULT_{consult}", "{}"])
                # ack may be delayed
                acked = rng.random() < 0.7
                if acked:
                    t_ack = t_req + pd.Timedelta(minutes=rng.uniform(10,45))
                    rows.append([t_ack, pid, eid, "specialist", consult, "CONSULT_ACK", "{}"])
                else:
                    t_ack = t_req + pd.Timedelta(minutes=rng.uniform(15,60))

                # Shift-aware activation for cath/endo
                if consult == "CARDIOLOGY":
                    delay_min = 30.0 if is_night(t_ack) else 15.0
                    t_start = t_ack + pd.Timedelta(minutes=delay_min + rng.normal(2.0, 2.0))
                    rows.append([t_start, pid, eid, "specialist", "CATH_LAB", "INTERVENTION_START_CATH", "{}"])
                if consult == "GASTRO":
                    delay_min = 30.0 if is_night(t_ack) else 15.0
                    t_start = t_ack + pd.Timedelta(minutes=delay_min + rng.normal(3.0, 3.0))
                    rows.append([t_start, pid, eid, "specialist", "ENDO_SUITE", "INTERVENTION_START_ENDO", "{}"])

            # Rounds
            for k in range(1,5):
                rows.append([t_hist + pd.Timedelta(minutes=15*k), pid, eid, "nurse", "ED", "NURSE_ROUND", "{}"])

    df = pd.DataFrame(rows, columns=["ts","patient_id","encounter_id","role","location","event_type","event_attrs"])
    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    return df.sort_values("ts").reset_index(drop=True)

# regenerate demo with shift-aware rules
events_demo = simulate_events(hours=24, base_rate=4.0, seed=101)
print("Simulated events (shift-aware):", len(events_demo))
events_demo.head(5)


In [ ]:

# --- Add follow-up activation skills with time thresholds ---
from datetime import timedelta

def minutes_since(ts_now: pd.Timestamp, ts_then: Optional[pd.Timestamp]) -> float:
    if ts_now is None or ts_then is None:
        return 0.0
    return float((ts_now - ts_then).total_seconds() / 60.0)

def skill_follow_up_cath_activation(state: WorkflowState):
    req_key = "REQUEST_CONSULT_CARDIOLOGY"
    if req_key in state.pending_orders and "INTERVENTION_START_CATH" not in state.completed_studies:
        t_req = state.event_times.get(req_key)
        # night threshold higher; day threshold lower
        thr = 25.0 if (state.now_ts is not None and (state.now_ts.hour >=22 or state.now_ts.hour <7)) else 18.0
        if minutes_since(state.now_ts, t_req) > thr:
            return {"action":"FOLLOW_UP_CATH_ACTIVATION", "reason":"Cath lab not started within expected window", "urgency":"high"}
    return None

def skill_follow_up_endo_activation(state: WorkflowState):
    req_key = "REQUEST_CONSULT_GASTRO"
    if req_key in state.pending_orders and "INTERVENTION_START_ENDO" not in state.completed_studies:
        t_req = state.event_times.get(req_key)
        thr = 25.0 if (state.now_ts is not None and (state.now_ts.hour >=22 or state.now_ts.hour <7)) else 18.0
        if minutes_since(state.now_ts, t_req) > thr:
            return {"action":"FOLLOW_UP_ENDO_ACTIVATION", "reason":"Endoscopy not started within expected window", "urgency":"high"}
    return None

# Extend SKILLS ordering: follow-ups early in list
try:
    SKILLS = [skill_follow_up_cath_activation, skill_follow_up_endo_activation] + SKILLS
except NameError:
    SKILLS = [skill_follow_up_cath_activation, skill_follow_up_endo_activation]

# Extend critic benefit/burden via instance-time defaults by monkey-patching init if present
if 'TinyCritics' in globals():
    orig_init = TinyCritics.__init__
    def _wrapped_init(self, *args, **kwargs):
        orig_init(self, *args, **kwargs)
        self.benefit_bias.update({"FOLLOW_UP_CATH_ACTIVATION": 1.0, "FOLLOW_UP_ENDO_ACTIVATION": 1.0})
        self.burden_bias.update({"FOLLOW_UP_CATH_ACTIVATION": 0.3, "FOLLOW_UP_ENDO_ACTIVATION": 0.3})
    TinyCritics.__init__ = _wrapped_init

# Update MetaController priors at instance creation time in user cell (below) by ensuring new actions are included


## P1 Patches — Med/Allergy Warnings, Pregnancy Adjustments, Troponin/ECG Compliance, SOP Search, Status Board

In [ ]:

# Safety: provide fallbacks if helpers are missing
def sanitize_event_attrs(s):
    import json
    try:
        if s is None or s=="" or s=="{}": return {}
        if isinstance(s, dict): return s
        return json.loads(s)
    except Exception:
        return {}

# Wrap update_state_from_event if present; else define minimal
if 'update_state_from_event' not in globals():
    def update_state_from_event(state, row): pass

_prev_update_fn = update_state_from_event
def update_state_from_event(state, row):
    _prev_update_fn(state, row)
    et = row.get("event_type","")
    attrs = sanitize_event_attrs(row.get("event_attrs","{}"))
    ts = pd.to_datetime(row["ts"], utc=True)

    if not hasattr(state, "event_times"): state.event_times = {}

    # Allergies list ingestion
    if et == "ALLERGY_LIST":
        if not hasattr(state, "allergies"): state.allergies = set()
        for a in attrs.get("allergies", []):
            state.allergies.add(str(a).lower())

    # Medication plan / pain plan (for warn-only intercepts)
    if et == "ANTIBIOTIC_PLAN":
        state.last_antibiotic_plan = attrs  # e.g., {"class":"cephalosporin", "name":"ceftriaxone"}
        state.event_times["ANTIBIOTIC_PLAN"] = ts
    if et == "PAIN_ORDER":
        state.last_pain_plan = attrs  # e.g., {"class":"NSAID","name":"ibuprofen"}
        state.event_times["PAIN_ORDER"] = ts

    # Pregnancy status
    if et == "PREGNANCY_STATUS":
        state.pregnant = bool(attrs.get("pregnant", False))

    # Symptom onset for chest pain
    if et == "SYMPTOM_ONSET":
        onset = attrs.get("onset_ts")
        if onset:
            try:
                state.symptom_onset_ts = pd.to_datetime(onset, utc=True)
            except Exception:
                state.symptom_onset_ts = ts
        else:
            state.symptom_onset_ts = ts

    # Troponin and ECG
    if et in ("TROPONIN_ORDER","ORDER_TROPONIN"):
        state.event_times["TROPONIN_T0"] = ts
    if et in ("TROPONIN_RESULT","TROPONIN_T1_RESULT"):
        state.event_times["TROPONIN_T1_RESULT"] = ts
    if et in ("ECG_RESULT","ECG_DONE"):
        state.ecg_count = getattr(state, "ecg_count", 0) + 1


In [ ]:

# Medication/Allergy warn-only skills
def skill_contrast_premed_checklist(state):
    ordered = state.event_times.get("ORDER_CT")
    has_result = state.event_times.get("CT_RESULT")
    if ordered and not has_result:
        if "contrast" in getattr(state, "allergies", set()):
            return {"action":"CONTRAST_PREMED_CHECKLIST",
                    "reason":"CT ordered & contrast allergy — follow premed protocol (warn-only)",
                    "urgency":"high"}
    return None

def skill_penicillin_ceph_warning(state):
    plan = getattr(state, "last_antibiotic_plan", {})
    if not plan: return None
    klass = str(plan.get("class","")).lower()
    if klass == "cephalosporin" and "penicillin" in getattr(state, "allergies", set()):
        return {"action":"WARN_PENICILLIN_CEPH_CROSSREACTIVITY",
                "reason":"Penicillin allergy + planned cephalosporin — consider alternative (warn-only)",
                "urgency":"high"}
    return None

def skill_nsaid_allergy_alternative(state):
    plan = getattr(state, "last_pain_plan", {})
    if not plan: return None
    klass = str(plan.get("class","")).lower()
    if klass == "nsaid" and ("nsaid" in getattr(state,"allergies",set()) or "aspirin" in getattr(state,"allergies",set())):
        return {"action":"SUGGEST_NON_NSAID_ANALGESIA",
                "reason":"NSAID allergy — suggest alternative pain management (warn-only)",
                "urgency":"medium"}
    return None


In [ ]:

# Pregnancy adjustments: age-adjusted D-dimer and stable imaging warning
def ddimer_threshold(age_years: int = None, pregnant: bool = False):
    if pregnant:
        return None  # show caution to follow local policy for pregnancy thresholds
    if age_years is None: return 500
    try:
        age = int(age_years)
    except Exception:
        return 500
    return age*10 if age > 50 else 500

def skill_pregnancy_imaging_warning(state):
    if getattr(state, "pregnant", False):
        ordered = state.event_times.get("ORDER_CT")
        has_result = state.event_times.get("CT_RESULT")
        if ordered and not has_result:
            return {"action":"PREGNANCY_IMAGING_CAUTION",
                    "reason":"Pregnancy flagged — verify imaging indications & alternatives (warn-only)",
                    "urgency":"medium"}
    return None


In [ ]:

# Troponin/ECG serial compliance skills
def _hours_since(ts, ref):
    if not ts or not ref: return None
    return (ts - ref).total_seconds()/3600.0

def skill_troponin_ecg_compliance(state):
    if not getattr(state, "chest_pain", False): return None
    now = state.now_ts
    t0 = state.event_times.get("TROPONIN_T0")
    t1 = state.event_times.get("TROPONIN_T1_RESULT")
    onset = getattr(state, "symptom_onset_ts", None)
    hours = _hours_since(now, onset) if onset else None
    if hours is not None and hours >= 6:
        return None  # single draw acceptable
    if t0 and not t1:
        mins = (now - t0).total_seconds()/60.0 if now else 0
        if mins >= 60:
            return {"action":"SERIAL_TROPONIN_REMINDER",
                    "reason":f"Symptoms <6h/unknown; T1 due (T0 + {mins:.0f} min)",
                    "urgency":"high"}
    return None

def skill_second_ecg_with_next_troponin(state):
    t0 = state.event_times.get("TROPONIN_T0")
    t1 = state.event_times.get("TROPONIN_T1_RESULT")
    if t0 and not t1 and "ECG_RESULT" in getattr(state,"completed_studies", set()):
        return {"action":"ECG_WITH_NEXT_TROP",
                "reason":"Bundle second ECG with next troponin to reduce workload",
                "urgency":"medium"}
    return None


In [ ]:

# SOP smart search + quick open
SOP_REGISTRY = [
    {"id":"SOP_CP_001","title":"Chest Pain Evaluation Protocol","tags":"chest pain ECG troponin risk HEART Marburg", "path":os.path.join(DATA_ROOT, "sops/chest_pain.pdf"},
    {"id":"SOP_SEPSIS_001","title":"Sepsis Recognition & Treatment Bundle","tags":"sepsis qSOFA lactate fluids antibiotics", "path":os.path.join(DATA_ROOT, "sops/sepsis_bundle.pdf"},
    {"id":"SOP_HF_AF_001","title":"Decompensated Heart Failure with Atrial Fibrillation","tags":"heart failure afib diuresis rate control echo", "path":os.path.join(DATA_ROOT, "sops/hf_afib.pdf"},
]

def sop_registry_load(csv_path=None):
    global SOP_REGISTRY
    if csv_path and os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        SOP_REGISTRY = df.to_dict(orient="records")
    return SOP_REGISTRY

def sop_search(query: str, top_k=3):
    q = str(query).lower()
    scored = []
    for r in SOP_REGISTRY:
        text = (r.get("title","") + " " + r.get("tags","")).lower()
        score = sum(1 for w in q.split() if w in text)
        if score>0: scored.append((score, r))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [r for _, r in scored[:top_k]]

def sop_quick_open(sop_id: str):
    for r in SOP_REGISTRY:
        if r["id"] == sop_id:
            return r.get("path")
    return None


In [ ]:

# Status board for devices
def status_board_df(resource, at_loc="ED"):
    rows = []
    for aid, meta in resource.assets.items():
        conf = 1.0
        if hasattr(resource, "location_confidence"):
            try: conf = float(resource.location_confidence(meta))
            except Exception: conf = 1.0
        if hasattr(resource, "_eta"):
            try: eta = float(resource._eta(meta.get("last_loc","ED"), at_loc, meta=meta))
            except Exception: eta = 8.0
        else:
            eta = LG.eta(meta.get("last_loc","ED"), at_loc)
        last_ts = meta.get("last_ts")
        last_seen_min = None
        if last_ts is not None:
            try:
                last_seen_min = (pd.Timestamp.utcnow().tz_localize("UTC") - last_ts).total_seconds()/60.0
            except Exception:
                last_seen_min = None
        rows.append({
            "asset_id": aid,
            "type": meta.get("type"),
            "last_loc": meta.get("last_loc"),
            "status": meta.get("status"),
            "battery_pct": meta.get("battery_pct", None),
            "eta_to_ED_min": round(float(eta),1),
            "confidence": round(float(conf),2),
            "verify_scan": conf < 0.5,
            "last_seen_min": None if last_seen_min is None else round(last_seen_min,1),
        })
    df = pd.DataFrame(rows).sort_values(["type","eta_to_ED_min","battery_pct"], ascending=[True, True, False])
    return df

def show_status_board():
    df = status_board_df(RESOURCE, at_loc="ED")
    try:
        from caas_jupyter_tools import display_dataframe_to_user
        display_dataframe_to_user("ED Status Board", df)
    except Exception:
        display(df)
    return df


In [ ]:

# Insert skills into SKILLS with sensible ordering
if 'SKILLS' not in globals(): SKILLS = []
_new = []
def _maybe_add(name):
    fn = globals().get(name)
    if fn and fn not in _new: _new.append(fn)
for name in ["skill_language_barrier_support","skill_language_cards","skill_scores_validation_prompt"]:
    _maybe_add(name)
for name in ["skill_chest_pain_atypical_bundle","skill_troponin_ecg_compliance","skill_second_ecg_with_next_troponin"]:
    _maybe_add(name)
for name in ["skill_contrast_premed_checklist","skill_penicillin_ceph_warning","skill_nsaid_allergy_alternative","skill_pregnancy_imaging_warning"]:
    _maybe_add(name)
for name in ["skill_page_cardiology_after_abnormal_ecg","skill_follow_up_imaging_overdue","skill_expedite_labs_if_deteriorating"]:
    _maybe_add(name)
for fn in SKILLS:
    if fn not in _new: _new.append(fn)
SKILLS = _new
print("P1 SKILLS installed (head):", [f.__name__ for f in SKILLS[:10]])


## CORE: Equipment Tracking (QR + Maintenance + Cleaning + Battery/Dock)

In [ ]:

import math, json, os, pandas as pd

# LocationGraph fallback if not present
if 'LocationGraph' not in globals():
    class LocationGraph:
        def __init__(self): self.edges = {}
        def add_edge(self, a,b,m):
            self.edges.setdefault(a,{})[b]=float(m)
            self.edges.setdefault(b,{})[a]=float(m)
        def eta(self, src, dst):
            if not src or not dst: return 8.0
            if src==dst: return 2.0
            import heapq
            dist={src:0.0}; pq=[(0.0,src)]
            while pq:
                d,u=heapq.heappop(pq)
                if u==dst: return max(1.0,min(30.0,d))
                if d>dist.get(u,1e9): continue
                for v,w in self.edges.get(u,{}).items():
                    nd=d+w
                    if nd<dist.get(v,1e9):
                        dist[v]=nd; heapq.heappush(pq,(nd,v))
            return 8.0
    LG = LocationGraph()
    LG.add_edge("ED","Room-1",2.0); LG.add_edge("ED","CT",4.0); LG.add_edge("Room-1","CT",3.0)

def _now_utc():
    return pd.Timestamp.utcnow().tz_localize("UTC")

class ResourceTracker:
    def __init__(self):
        self.assets = {}     # asset_id -> meta
        self.holds = {}      # asset_id -> {encounter_id, expires_ts, priority}
    # --- Lifecyle ---
    def ingest_seed(self, asset_id, typ="ultrasound", loc="ED", battery=100, docked=True):
        self.assets[asset_id] = {
            "type": typ, "last_loc": loc, "battery_pct": int(battery), "status":"available",
            "last_ts": _now_utc(), "maintenance":"ok", "needs_cleaning": False, "docked": bool(docked)
        }
    # --- Updates from events ---
    def update_from_event(self, row: pd.Series):
        et = row.get("event_type","")
        attrs = row.get("event_attrs","{}")
        if isinstance(attrs,str):
            try: attrs = json.loads(attrs) if attrs and attrs!="{}" else {}
            except Exception: attrs = {}
        loc = row.get("location","ED")
        ts = pd.to_datetime(row.get("ts", _now_utc()), utc=True)
        if et == "QR_SCAN":
            aid = attrs.get("asset_id"); typ = attrs.get("type","ultrasound")
            if not aid: return
            self.assets.setdefault(aid, {"type":typ,"last_loc":loc,"battery_pct":100,"status":"available","maintenance":"ok","needs_cleaning":False,"docked":True})
            self.assets[aid]["last_loc"]=loc; self.assets[aid]["last_ts"]=ts
            if "battery_pct" in attrs: self.assets[aid]["battery_pct"]=int(attrs["battery_pct"])
            if "docked" in attrs: self.assets[aid]["docked"]=bool(attrs["docked"])
        elif et == "MAINTENANCE_REPORTED":
            aid = attrs.get("asset_id")
            if aid and aid in self.assets:
                self.assets[aid]["maintenance"]="broken"; self.assets[aid]["status"]="unavailable"; self.assets[aid]["last_ts"]=ts
        elif et == "MAINTENANCE_RESOLVED":
            aid = attrs.get("asset_id")
            if aid and aid in self.assets:
                self.assets[aid]["maintenance"]="ok"; self.assets[aid]["status"]="available"; self.assets[aid]["last_ts"]=ts
        elif et == "CLEANING_REQUIRED":
            aid = attrs.get("asset_id")
            if aid and aid in self.assets:
                self.assets[aid]["needs_cleaning"]=True; self.assets[aid]["last_ts"]=ts
        elif et == "CLEANING_DONE":
            aid = attrs.get("asset_id")
            if aid and aid in self.assets:
                self.assets[aid]["needs_cleaning"]=False; self.assets[aid]["docked"]=bool(attrs.get("docked",True)); self.assets[aid]["last_ts"]=ts
        elif et == "ULTRASOUND_STARTED":
            aid = attrs.get("asset_id")
            if aid and aid in self.assets: self.assets[aid]["status"]="in_use"; self.assets[aid]["last_ts"]=ts
        elif et == "ULTRASOUND_DONE":
            aid = attrs.get("asset_id")
            if aid and aid in self.assets: self.assets[aid]["status"]="available"; self.assets[aid]["last_ts"]=ts
    # --- Usability & confidence ---
    def is_usable(self, meta):
        return meta.get("status")=="available" and meta.get("maintenance")=="ok" and not meta.get("needs_cleaning", False)
    def last_seen_minutes(self, meta):
        ts = meta.get("last_ts"); 
        if not ts: return 999.0
        return max(0.0, (_now_utc() - ts).total_seconds()/60.0)
    def location_confidence(self, meta, half_life=15.0):
        mins = self.last_seen_minutes(meta)
        return math.exp(-math.log(2.0)*mins/half_life)  # 1.0 fresh -> 0.0 stale
    def _eta(self, loc_from: str, loc_to: str, meta=None):
        base = LG.eta(loc_from or "ED", loc_to or "ED")
        conf = self.location_confidence(meta) if meta else 1.0
        penalty = (1.0 - conf) * 6.0
        return base + penalty
    def closest(self, asset_type='ultrasound', at_loc='ED'):
        best=None
        for aid, meta in self.assets.items():
            if meta.get("type")!=asset_type or not self.is_usable(meta): continue
            eta = self._eta(meta.get("last_loc","ED"), at_loc, meta=meta)
            cand={"asset_id":aid,"last_loc":meta.get("last_loc"),"eta_min":round(float(eta),1),"battery_pct":meta.get("battery_pct",100),"conf":round(self.location_confidence(meta),2)}
            if (best is None) or (eta < best["eta_min"]) or (eta == best["eta_min"] and cand["battery_pct"] > best.get("battery_pct",0)):
                best=cand
        return best

# Global instance for core architecture
if 'RESOURCE' not in globals():
    RESOURCE = ResourceTracker()

# QR generator: produce QR PNGs for asset IDs (offline)
def generate_equipment_qr_codes(asset_ids, out_dir=os.path.join(DATA_ROOT, "qr")):
    try:
        import qrcode
    except Exception:
        import sys, subprocess
        subprocess.check_call([sys.executable,"-m","pip","install","qrcode[pil]"], stdout=subprocess.DEVNULL)
        import qrcode
    os.makedirs(out_dir, exist_ok=True)
    files=[]
    for aid in asset_ids:
        img = qrcode.make(aid)
        path = os.path.join(out_dir, f"{aid}.png")
        img.save(path)
        files.append(path)
    return files

# Status board helpers
def status_board_df(resource: ResourceTracker, at_loc="ED"):
    rows=[]
    for aid, meta in resource.assets.items():
        eta = resource._eta(meta.get("last_loc","ED"), at_loc, meta=meta)
        conf = resource.location_confidence(meta)
        rows.append({
            "asset_id": aid, "type": meta.get("type"), "last_loc": meta.get("last_loc"),
            "status": meta.get("status"), "battery_pct": meta.get("battery_pct"),
            "eta_to_ED_min": round(float(eta),1), "confidence": round(float(conf),2),
            "verify_scan": conf < 0.5, "docked": meta.get("docked", True),
            "needs_cleaning": meta.get("needs_cleaning", False), "maintenance": meta.get("maintenance","ok")
        })
    df = pd.DataFrame(rows).sort_values(["type","eta_to_ED_min","battery_pct"], ascending=[True, True, False])
    return df

def show_status_board():
    df = status_board_df(RESOURCE, at_loc="ED")
    try:
        from caas_jupyter_tools import display_dataframe_to_user
        display_dataframe_to_user("ED Status Board", df)
    except Exception:
        display(df)
    return df


## CORE: SOP Access (Registry + Search + Quick Open)

In [ ]:

import pandas as pd, os, json

SOP_REGISTRY = [
    {"id":"SOP_CP_001","title":"Chest Pain Evaluation Protocol","tags":"chest pain ECG troponin risk HEART Marburg","path":os.path.join(DATA_ROOT, "sops/chest_pain.pdf"},
    {"id":"SOP_SEPSIS_001","title":"Sepsis Recognition & Treatment Bundle","tags":"sepsis qSOFA lactate fluids antibiotics","path":os.path.join(DATA_ROOT, "sops/sepsis_bundle.pdf"},
    {"id":"SOP_HF_AF_001","title":"Decompensated Heart Failure with Atrial Fibrillation","tags":"heart failure afib diuresis rate control echo","path":os.path.join(DATA_ROOT, "sops/hf_afib.pdf"},
]

def sop_registry_load(csv_path=None):
    global SOP_REGISTRY
    if csv_path and os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        SOP_REGISTRY = df.to_dict(orient="records")
    return SOP_REGISTRY

def sop_search(query: str, top_k=3):
    q = str(query).lower()
    scored=[]
    for r in SOP_REGISTRY:
        text = (r.get("title","")+" "+r.get("tags","")).lower()
        score = sum(1 for w in q.split() if w in text)
        if score>0: scored.append((score, r))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [r for _, r in scored[:top_k]]

def sop_quick_open(sop_id: str):
    for r in SOP_REGISTRY:
        if r["id"] == sop_id:
            return r.get("path")
    return None

# Optional skill to surface SOPs based on events
def skill_surface_relevant_sops(state):
    if getattr(state, "chest_pain", False):
        hits = sop_search("chest pain ECG troponin risk", top_k=1)
        if hits:
            return {"action":"OPEN_SOP","reason":"Relevant SOP for chest pain","attrs":{"sop_id": hits[0]["id"], "path": hits[0]["path"]}, "urgency":"info"}
    # Sepsis hold scenario (qSOFA-like >=2)
    risk = 0
    if getattr(state,"sbp",None) is not None and state.sbp<=100: risk+=1
    if getattr(state,"rr",None) is not None and state.rr>=22: risk+=1
    if getattr(state,"ams",False): risk+=1
    if risk>=2:
        hits = sop_search("sepsis bundle", top_k=1)
        if hits:
            return {"action":"OPEN_SOP","reason":"Relevant SOP for suspected sepsis","attrs":{"sop_id": hits[0]["id"], "path": hits[0]["path"]}, "urgency":"info"}
    return None


## CORE: Troponin Delta (Display-only) with Coordination Nudge

In [ ]:

# Extend state to store troponin numeric values when T0/T1 results arrive
_prev_update_td = update_state_from_event
def update_state_from_event(state: WorkflowState, row):
    _prev_update_td(state, row)
    et = row.get("event_type","")
    attrs = row.get("event_attrs","{}")
    if isinstance(attrs, str):
        try: attrs = json.loads(attrs) if attrs and attrs!="{}" else {}
        except Exception: attrs = {}
    if et in ("TROPONIN_RESULT","TROPONIN_T0_RESULT"):
        try: state.trop_t0_val = float(attrs.get("value"))
        except: pass
    if et in ("TROPONIN_T1_RESULT","TROPONIN_DELTA_RESULT"):
        try: state.trop_t1_val = float(attrs.get("value"))
        except: pass

def compute_troponin_delta(state):
    v0 = getattr(state, "trop_t0_val", None)
    v1 = getattr(state, "trop_t1_val", None)
    if v0 is None or v1 is None: return None
    delta_abs = v1 - v0
    if v0 == 0: delta_pct = float('inf') if v1>0 else 0.0
    else: delta_pct = (v1 - v0) / abs(v0) * 100.0
    return {"v0": v0, "v1": v1, "delta_abs": delta_abs, "delta_pct": delta_pct}

# Config thresholds for display-only review prompts
CONFIG.setdefault("troponin", {"delta_abs_ngL": 5.0, "delta_pct": 20.0})

def skill_review_troponin_trend(state):
    td = compute_troponin_delta(state)
    if not td: return None
    thr_abs = CONFIG["troponin"]["delta_abs_ngL"]
    thr_pct = CONFIG["troponin"]["delta_pct"]
    if abs(td["delta_abs"]) >= thr_abs or abs(td["delta_pct"]) >= thr_pct:
        # Coordination nudge only — no diagnosis
        reason = f"Troponin trend: Δ {td['delta_abs']:+.1f} ng/L ({td['delta_pct']:+.0f}%) — review trend"
        return {"action":"REVIEW_TROPONIN_TREND","reason": reason, "urgency":"medium"}
    return None


## Wire-in Core Skills (SOP surfacing + Troponin delta)

In [ ]:

if 'SKILLS' not in globals(): SKILLS = []
_new = []
def _maybe_add(name):
    fn = globals().get(name)
    if fn and fn not in _new: _new.append(fn)

# Keep language + validation first
for name in ["skill_language_barrier_support","skill_language_cards","skill_scores_validation_prompt"]:
    _maybe_add(name)

# Core workflow skills
for name in ["skill_chest_pain_atypical_bundle","skill_troponin_ecg_compliance","skill_second_ecg_with_next_troponin"]:
    _maybe_add(name)

# SOP surfacing & troponin delta (display-only)
for name in ["skill_surface_relevant_sops","skill_review_troponin_trend"]:
    _maybe_add(name)

# Existing P1 skills (allergy/pregnancy warn-only) and coordination skills
for name in ["skill_contrast_premed_checklist","skill_penicillin_ceph_warning","skill_nsaid_allergy_alternative","skill_pregnancy_imaging_warning",
             "skill_page_cardiology_after_abnormal_ecg","skill_follow_up_imaging_overdue","skill_expedite_labs_if_deteriorating"]:
    _maybe_add(name)

# Append the rest unchanged
for fn in SKILLS:
    if fn not in _new: _new.append(fn)
SKILLS = _new
print("Core skills wired (head):", [f.__name__ for f in SKILLS[:10]])


## CORE: Equipment UI Bridge (state snapshots + action handlers)

In [ ]:

# Extend ResourceTracker with reservation methods and TTL cleanup (idempotent if already present)
from datetime import timedelta

def _rt_reserve(self, asset_id: str, by="ED-OPS", ttl_minutes=10):
    meta = self.assets.get(asset_id)
    if not meta: return False, "unknown_asset"
    if meta.get("status") in ("in_use","unavailable"): return False, "not_reservable"
    # set reservation
    until = _now_utc() + pd.Timedelta(minutes=int(ttl_minutes))
    meta["reservation"] = {"by": by, "expires_at": until}
    meta["status"] = "reserved"
    return True, "ok"

def _rt_release(self, asset_id: str):
    meta = self.assets.get(asset_id)
    if not meta: return False, "unknown_asset"
    meta.pop("reservation", None)
    if meta.get("status") == "reserved":
        meta["status"] = "available"
    return True, "ok"

def _rt_cleanup(self):
    now = _now_utc()
    for aid, meta in list(self.assets.items()):
        r = meta.get("reservation")
        if r and isinstance(r, dict):
            try:
                exp = r.get("expires_at")
                if exp and now >= exp:
                    meta.pop("reservation", None)
                    if meta.get("status") == "reserved":
                        meta["status"] = "available"
            except Exception:
                pass

# Monkey-patch only if missing
if not hasattr(ResourceTracker, "reserve"): ResourceTracker.reserve = _rt_reserve
if not hasattr(ResourceTracker, "release"): ResourceTracker.release = _rt_release
if not hasattr(ResourceTracker, "cleanup"): ResourceTracker.cleanup = _rt_cleanup

# --- UI Bridge: snapshot writers and action applier ---
import os, json

UI_STATE_DIR = os.path.join(DATA_ROOT, "ui")

def ui_status_board_json(resource=RESOURCE, at_loc="ED", out_dir=UI_STATE_DIR):
    os.makedirs(out_dir, exist_ok=True)
    RESOURCE.cleanup()
    df = status_board_df(resource, at_loc=at_loc)
    data = {
        "generated_at": pd.Timestamp.utcnow().isoformat() + "Z",
        "equipment": df.to_dict(orient="records")
    }
    out_path = f"{out_dir}/equipment.json"
    with open(out_path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print("Wrote:", out_path, "items:", len(data["equipment"]))
    return data

def ui_apply_action(action: str, payload: dict):
    # Map UI intents to tracker updates
    a = action.lower()
    aid = payload.get("asset_id")
    if a == "reserve":
        ok, msg = RESOURCE.reserve(aid, by=payload.get("by","ED-OPS"), ttl_minutes=payload.get("ttl",10))
    elif a == "release":
        ok, msg = RESOURCE.release(aid)
    elif a == "verify_scan":
        # payload: asset_id, location, battery_pct?, docked?
        row = pd.Series({
            "event_type": "QR_SCAN",
            "ts": pd.Timestamp.utcnow().isoformat() + "Z",
            "location": payload.get("location","ED"),
            "event_attrs": json.dumps({
                "asset_id": aid,
                "battery_pct": payload.get("battery_pct", None),
                "docked": payload.get("docked", None),
                "type": payload.get("type","ultrasound")
            })
        })
        RESOURCE.update_from_event(row)
        ok, msg = True, "ok"
    elif a == "report_broken":
        row = pd.Series({"event_type":"MAINTENANCE_REPORTED","ts":pd.Timestamp.utcnow().isoformat()+"Z","location":"ED","event_attrs":json.dumps({"asset_id":aid})})
        RESOURCE.update_from_event(row); ok, msg = True, "ok"
    elif a == "resolve_maintenance":
        row = pd.Series({"event_type":"MAINTENANCE_RESOLVED","ts":pd.Timestamp.utcnow().isoformat()+"Z","location":"ED","event_attrs":json.dumps({"asset_id":aid})})
        RESOURCE.update_from_event(row); ok, msg = True, "ok"
    elif a == "mark_cleaned":
        row = pd.Series({"event_type":"CLEANING_DONE","ts":pd.Timestamp.utcnow().isoformat()+"Z","location":"ED","event_attrs":json.dumps({"asset_id":aid,"docked": True})})
        RESOURCE.update_from_event(row); ok, msg = True, "ok"
    elif a == "toggle_docked":
        meta = RESOURCE.assets.get(aid, {})
        row = pd.Series({"event_type":"CLEANING_DONE","ts":pd.Timestamp.utcnow().isoformat()+"Z","location":"ED","event_attrs":json.dumps({"asset_id":aid,"docked": not bool(meta.get("docked", True))})})
        RESOURCE.update_from_event(row); ok, msg = True, "ok"
    else:
        ok, msg = False, "unknown_action"
    # Always refresh snapshot after an action
    ui_status_board_json()
    return {"ok": ok, "msg": msg}

# Convenience cell to seed a few assets and write first snapshot
def ui_seed_and_snapshot():
    if not RESOURCE.assets:
        RESOURCE.ingest_seed("US_01","ultrasound","Room 3",battery=85,docked=False)
        RESOURCE.ingest_seed("US_02","ultrasound","Hallway B",battery=22,docked=False)
        RESOURCE.ingest_seed("CRASH_01","crash_cart","Resus 1",battery=100,docked=True)
    return ui_status_board_json()


## UI Artifact: write EDEquipmentTrackerPro.tsx to disk

In [ ]:

pro_path = os.path.join(DATA_ROOT, "EDEquipmentTrackerPro.tsx")
os.makedirs(os.path.dirname(pro_path), exist_ok=True)
# If already written earlier, keep it; else write a copy of the improved UI
try:
    with open(pro_path, "r") as f:
        print("UI already present at", pro_path)
except FileNotFoundError:
    ui_src = """
import React, { useMemo, useState } from "react";
import {
  QrCode, MapPin, Clock, Battery, Search, Scan, AlertTriangle, Wrench,
  ShieldAlert, Power, CheckCircle2, XCircle, Timer, RefreshCcw, PlugZap, Lock, Unlock
} from "lucide-react";

/**
 * ED Equipment Tracker (Pro)
 * - Confidence & stale-data warnings (verify-scan CTA)
 * - Maintenance & cleaning lifecycle
 * - Docked/charging & low-battery warnings
 * - Reservation with TTL + conflict handling
 * - Filters (type/status/critical) + sort by ETA, then battery
 * - Simulated QR scan + location update
 * TailwindCSS + lucide-react
 */

type Status = "Available" | "In Use" | "Reserved" | "Missing" | "Unavailable" | "Ready";
type Maintenance = "ok" | "broken";

type Equipment = {
  id: string;
  type: string;
  location: string;
  status: Status;
  battery: number | null;
  lastSeenMin: number;         // minutes since last seen
  confidence: number;          // 0..1
  needsCleaning: boolean;
  maintenance: Maintenance;
  docked: boolean;
  reservation?: { by: string; expiresAt: number }; // epoch ms
};

const LOW_BATT = 30;
const STALE_MIN = 30;
const DEFAULT_ETA = 5;

const ALL_TYPES = ["All", "Ultrasound", "Crash Cart", "Wheelchair", "IV Pump", "Monitor"] as const;
const ALL_STATUS = ["Any", "Available", "In Use", "Reserved", "Missing", "Unavailable", "Ready"] as const;

function etaMinutes(from: string, to: string, confidence: number): number {
  if (!from || !to) return DEFAULT_ETA;
  if (from === to) return 2;
  // one-floor ED heuristic; add penalty for low confidence
  const base = DEFAULT_ETA;
  const penalty = (1 - Math.max(0, Math.min(1, confidence))) * 6;
  return Math.round((base + penalty) * 10) / 10;
}

function statusColor(s: Status) {
  switch (s) {
    case "Available": return "text-green-700 bg-green-100";
    case "In Use": return "text-blue-700 bg-blue-100";
    case "Reserved": return "text-amber-700 bg-amber-100";
    case "Missing": return "text-red-700 bg-red-100";
    case "Unavailable": return "text-gray-700 bg-gray-200";
    case "Ready": return "text-green-700 bg-green-100";
    default: return "text-gray-700 bg-gray-100";
  }
}

function battColor(b: number | null) {
  if (b === null) return "text-gray-400";
  if (b > 50) return "text-green-700";
  if (b > LOW_BATT) return "text-amber-700";
  return "text-red-700";
}

function chip({ text, className }: { text: string; className?: string }) {
  return <span className={`px-2 py-0.5 rounded text-xs font-medium ${className || ""}`}>{text}</span>;
}

function now() { return Date.now(); }

export default function EDEquipmentTrackerPro() {
  const [term, setTerm] = useState("");
  const [typeFilter, setTypeFilter] = useState<typeof ALL_TYPES[number]>("All");
  const [statusFilter, setStatusFilter] = useState<typeof ALL_STATUS[number]>("Any");
  const [criticalOnly, setCriticalOnly] = useState(false);
  const [scannerOpen, setScannerOpen] = useState(false);
  const [scanId, setScanId] = useState<string>("");
  const [scanLoc, setScanLoc] = useState<string>("");

  const [equipment, setEquipment] = useState<Equipment[]>([
    { id: "US_01", type: "Ultrasound", location: "Room 3", status: "In Use", battery: 85, lastSeenMin: 2, confidence: 0.95, needsCleaning: false, maintenance: "ok", docked: false },
    { id: "US_02", type: "Ultrasound", location: "Hallway B", status: "Available", battery: 22, lastSeenMin: 45, confidence: 0.35, needsCleaning: true, maintenance: "ok", docked: false },
    { id: "US_03", type: "Ultrasound", location: "Unknown", status: "Missing", battery: null, lastSeenMin: 120, confidence: 0.15, needsCleaning: false, maintenance: "ok", docked: false },
    { id: "CRASH_01", type: "Crash Cart", location: "Resus 1", status: "Ready", battery: 100, lastSeenMin: 5, confidence: 0.98, needsCleaning: false, maintenance: "ok", docked: true },
    { id: "CRASH_02", type: "Crash Cart", location: "Room 8", status: "In Use", battery: 90, lastSeenMin: 1, confidence: 0.99, needsCleaning: false, maintenance: "ok", docked: false },
    { id: "WC_01", type: "Wheelchair", location: "Main Entrance", status: "Available", battery: null, lastSeenMin: 30, confidence: 0.70, needsCleaning: false, maintenance: "ok", docked: false },
    { id: "IV_01", type: "IV Pump", location: "Room 2", status: "In Use", battery: 45, lastSeenMin: 8, confidence: 0.88, needsCleaning: false, maintenance: "ok", docked: false },
    { id: "MON_01", type: "Monitor", location: "Hallway A", status: "Available", battery: 75, lastSeenMin: 12, confidence: 0.85, needsCleaning: false, maintenance: "ok", docked: true },
  ]);

  const LOCATIONS = ["ED", "Room 1", "Room 2", "Room 3", "Room 4", "Room 5", "Room 6", "Room 7", "Room 8", "Resus 1", "Resus 2", "Hallway A", "Hallway B", "Main Entrance", "Triage", "Storage"];

  function update(id: string, patch: Partial<Equipment>) {
    setEquipment(prev => prev.map(e => (e.id === id ? { ...e, ...patch } : e)));
  }

  function reserve(id: string, by = "ED-OPS", ttlMin = 10) {
    setEquipment(prev => prev.map(e => {
      if (e.id !== id) return e;
      if (e.status === "In Use" || e.status === "Reserved" || e.maintenance === "broken") return e; // reject conflicts
      return { ...e, status: "Reserved", reservation: { by, expiresAt: now() + ttlMin * 60_000 } };
    }));
  }
  function release(id: string) { update(id, { status: "Available", reservation: undefined }); }

  function verifyScan(id: string, newLoc: string) {
    update(id, { location: newLoc, lastSeenMin: 0, confidence: 0.99, status: (prevOf(id)?.status === "Missing" ? "Available" : prevOf(id)?.status) || "Available" });
  }
  function prevOf(id: string) { return equipment.find(e => e.id === id); }

  function reportBroken(id: string) { update(id, { maintenance: "broken", status: "Unavailable" }); }
  function resolveMaintenance(id: string) { update(id, { maintenance: "ok", status: "Available" }); }
  function markCleaned(id: string) { update(id, { needsCleaning: false }); }
  function toggleDocked(id: string) { update(id, { docked: !(prevOf(id)?.docked ?? false) }); }

  // Expire reservations
  const _ = useMemo(() => {
    const nowMs = now();
    const changed: string[] = [];
    equipment.forEach(e => {
      if (e.reservation && e.reservation.expiresAt <= nowMs) {
        changed.push(e.id);
      }
    });
    if (changed.length) {
      setEquipment(prev => prev.map(e => e.reservation && e.reservation.expiresAt <= nowMs ? { ...e, reservation: undefined, status: "Available" } : e));
    }
    // eslint-disable-next-line react-hooks/exhaustive-deps
  }, [equipment.map(e => e.reservation?.expiresAt).join(",")]);

  const filtered = useMemo(() => {
    return equipment
      .filter(e => {
        const txt = `${e.id} ${e.type} ${e.location}`.toLowerCase();
        const matchesTerm = term.trim() === "" || txt.includes(term.toLowerCase());
        const matchesType = typeFilter === "All" || e.type === typeFilter;
        const matchesStatus = statusFilter === "Any" || e.status === statusFilter;
        const critical = (e.confidence < 0.5) || (e.lastSeenMin > STALE_MIN) || (typeof e.battery === "number" && e.battery <= LOW_BATT) || e.maintenance === "broken" || e.needsCleaning;
        const matchesCritical = !criticalOnly || critical;
        return matchesTerm && matchesType && matchesStatus && matchesCritical;
      })
      .map(e => ({
        ...e,
        eta: etaMinutes(e.location, "ED", e.confidence),
      }))
      .sort((a, b) => (a.eta - b.eta) || ((b.battery ?? -1) - (a.battery ?? -1)));
  }, [equipment, term, typeFilter, statusFilter, criticalOnly]);

  function simulateScan() {
    const ids = equipment.map(e => e.id);
    const rnd = ids[Math.floor(Math.random() * ids.length)];
    setScanId(rnd);
  }

  return (
    <div className="max-w-7xl mx-auto p-6 bg-gray-50 min-h-screen">
      <div className="bg-white rounded-2xl shadow">
        {/* Header */}
        <div className="p-6 border-b border-gray-200 flex items-center justify-between">
          <div className="flex items-center gap-3">
            <QrCode className="w-8 h-8 text-blue-600" />
            <div>
              <h1 className="text-2xl font-bold text-gray-900">ED Equipment Tracker</h1>
              <p className="text-sm text-gray-600">Locate, reserve, and maintain shared devices</p>
            </div>
          </div>
          <div className="flex items-center gap-3">
            <button
              onClick={() => setScannerOpen(true)}
              className="flex items-center gap-2 bg-blue-600 text-white px-4 py-2 rounded-lg hover:bg-blue-700"
            >
              <Scan className="w-4 h-4" />
              Scan QR
            </button>
          </div>
        </div>

        {/* Filters */}
        <div className="p-6 bg-gray-50 border-b border-gray-200">
          <div className="grid grid-cols-1 md:grid-cols-4 gap-3">
            <div className="relative col-span-2">
              <Search className="absolute left-3 top-1/2 -translate-y-1/2 text-gray-400 w-4 h-4" />
              <input
                className="pl-10 pr-3 py-2 w-full border border-gray-300 rounded-lg focus:ring-2 focus:ring-blue-500 focus:border-transparent"
                placeholder="Search by ID, type, or location..."
                value={term}
                onChange={e => setTerm(e.target.value)}
              />
            </div>
            <select
              className="px-3 py-2 border border-gray-300 rounded-lg focus:ring-2 focus:ring-blue-500"
              value={typeFilter}
              onChange={e => setTypeFilter(e.target.value as any)}
            >
              {ALL_TYPES.map(t => <option key={t} value={t}>{t}</option>)}
            </select>
            <div className="flex items-center gap-2">
              <select
                className="flex-1 px-3 py-2 border border-gray-300 rounded-lg focus:ring-2 focus:ring-blue-500"
                value={statusFilter}
                onChange={e => setStatusFilter(e.target.value as any)}
              >
                {ALL_STATUS.map(s => <option key={s} value={s}>{s}</option>)}
              </select>
              <label className="flex items-center gap-2 text-sm text-gray-700 select-none">
                <input type="checkbox" checked={criticalOnly} onChange={e => setCriticalOnly(e.target.checked)} />
                Critical only
              </label>
            </div>
          </div>
        </div>

        {/* Grid */}
        <div className="p-6 grid grid-cols-1 md:grid-cols-2 lg:grid-cols-3 gap-4">
          {filtered.map(e => {
            const isLowBatt = typeof e.battery === "number" && e.battery <= LOW_BATT;
            const isStale = e.lastSeenMin > STALE_MIN || e.confidence < 0.5;
            const canReserve = e.status === "Available" && e.maintenance === "ok";
            return (
              <div key={e.id} className="border border-gray-200 rounded-xl p-4 hover:shadow-md transition-shadow">
                <div className="flex items-start justify-between">
                  <div>
                    <h3 className="font-semibold text-lg text-gray-900">{e.id}</h3>
                    <p className="text-sm text-gray-600">{e.type}</p>
                  </div>
                  {chip({ text: e.status, className: statusColor(e.status) })}
                </div>

                <div className="mt-3 space-y-2">
                  <div className="flex items-center gap-2 text-sm">
                    <MapPin className="w-4 h-4 text-gray-400" />
                    <span>{e.location}</span>
                    {isStale && <AlertTriangle className="w-4 h-4 text-amber-500" title="Location likely stale — verify scan" />}
                    {e.docked && <PlugZap className="w-4 h-4 text-green-600" title="Docked/charging" />}
                  </div>

                  <div className="flex items-center gap-2 text-sm text-gray-600">
                    <Clock className="w-4 h-4 text-gray-400" />
                    <span>Last seen: {e.lastSeenMin} min</span>
                    <Timer className="w-4 h-4 text-gray-400 ml-3" />
                    <span>ETA to ED: {e.eta} min</span>
                  </div>

                  {e.battery !== null && (
                    <div className="flex items-center gap-2 text-sm">
                      <Battery className={`w-4 h-4 ${battColor(e.battery)}`} />
                      <span className={`${battColor(e.battery)} font-medium`}>
                        {e.battery}% {isLowBatt && "(low)"}
                      </span>
                    </div>
                  )}

                  <div className="flex flex-wrap gap-2 text-xs">
                    {e.needsCleaning && chip({ text: "Needs cleaning", className: "text-amber-700 bg-amber-100" })}
                    {e.maintenance === "broken" && chip({ text: "Broken", className: "text-red-700 bg-red-100" })}
                    {isStale && chip({ text: `Confidence ${Math.round(e.confidence*100)}%`, className: "text-amber-700 bg-amber-100" })}
                    {e.reservation && chip({ text: `Reserved until ${new Date(e.reservation.expiresAt).toLocaleTimeString()}`, className: "text-amber-700 bg-amber-100" })}
                  </div>
                </div>

                {/* Actions */}
                <div className="mt-3 pt-3 border-t border-gray-100 grid grid-cols-2 gap-2">
                  <button
                    onClick={() => setScannerOpen(true) || setScanId(e.id)}
                    className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg hover:bg-gray-50"
                    title="Verify location via scan"
                  >
                    <QrCode className="w-4 h-4" /> Verify scan
                  </button>

                  {canReserve ? (
                    <button
                      onClick={() => reserve(e.id)}
                      className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg hover:bg-gray-50"
                      title="Reserve for 10 minutes"
                    >
                      <Lock className="w-4 h-4" /> Reserve
                    </button>
                  ) : e.status === "Reserved" ? (
                    <button
                      onClick={() => release(e.id)}
                      className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg hover:bg-gray-50"
                      title="Release reservation"
                    >
                      <Unlock className="w-4 h-4" /> Release
                    </button>
                  ) : (
                    <button
                      disabled
                      className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg opacity-50 cursor-not-allowed"
                      title="Not reservable now"
                    >
                      <Lock className="w-4 h-4" /> Reserve
                    </button>
                  )}

                  {e.needsCleaning ? (
                    <button
                      onClick={() => markCleaned(e.id)}
                      className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg hover:bg-gray-50"
                    >
                      <CheckCircle2 className="w-4 h-4 text-green-600" /> Mark cleaned
                    </button>
                  ) : (
                    <button disabled className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg opacity-50 cursor-not-allowed">
                      <CheckCircle2 className="w-4 h-4" /> Mark cleaned
                    </button>
                  )}

                  {e.maintenance === "ok" ? (
                    <button
                      onClick={() => reportBroken(e.id)}
                      className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg hover:bg-gray-50"
                    >
                      <ShieldAlert className="w-4 h-4 text-red-600" /> Report broken
                    </button>
                  ) : (
                    <button
                      onClick={() => resolveMaintenance(e.id)}
                      className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg hover:bg-gray-50"
                    >
                      <Wrench className="w-4 h-4" /> Resolve maintenance
                    </button>
                  )}

                  <button
                    onClick={() => toggleDocked(e.id)}
                    className="flex items-center justify-center gap-2 px-3 py-2 border rounded-lg hover:bg-gray-50 col-span-2"
                  >
                    <Power className="w-4 h-4" /> {e.docked ? "Undock" : "Dock/charge"}
                  </button>
                </div>
              </div>
            );
          })}
        </div>

        {/* Scanner Modal */}
        {scannerOpen && (
          <div className="fixed inset-0 bg-black/50 flex items-center justify-center z-50">
            <div className="bg-white rounded-xl p-6 w-full max-w-md">
              <h3 className="text-lg font-semibold mb-4">QR Code Scanner</h3>
              <div className="p-8 border-2 border-dashed rounded-lg text-center text-gray-600 mb-4">
                <QrCode className="w-16 h-16 mx-auto mb-2 text-gray-400" />
                <p>Position QR code in camera view</p>
                <button className="text-blue-600 text-sm underline mt-2" onClick={simulateScan}>
                  Simulate scan
                </button>
              </div>
              <div className="grid gap-3 mb-4">
                <div>
                  <label className="block text-sm font-medium mb-1">Scanned ID</label>
                  <input
                    className="w-full border rounded-lg px-3 py-2"
                    placeholder="US_01"
                    value={scanId}
                    onChange={e => setScanId(e.target.value)}
                  />
                </div>
                <div>
                  <label className="block text-sm font-medium mb-1">New location</label>
                  <select className="w-full border rounded-lg px-3 py-2" value={scanLoc} onChange={e => setScanLoc(e.target.value)}>
                    <option value="">Select location...</option>
                    {LOCATIONS.map(l => <option key={l} value={l}>{l}</option>)}
                  </select>
                </div>
              </div>
              <div className="flex items-center gap-3">
                <button
                  onClick={() => {
                    if (scanId && scanLoc) {
                      verifyScan(scanId, scanLoc);
                      setScannerOpen(false);
                      setScanId(""); setScanLoc("");
                    }
                  }}
                  className="flex-1 bg-blue-600 text-white px-4 py-2 rounded-lg hover:bg-blue-700"
                  disabled={!scanId || !scanLoc}
                >
                  Update location
                </button>
                <button className="flex-1 bg-gray-200 text-gray-800 px-4 py-2 rounded-lg hover:bg-gray-300" onClick={() => { setScannerOpen(false); setScanId(""); setScanLoc(""); }}>
                  Cancel
                </button>
              </div>
            </div>
          </div>
        )}
      </div>

      {/* Footer / Legend */}
      <div className="text-xs text-gray-500 mt-4 flex items-center gap-4">
        <div className="flex items-center gap-1"><AlertTriangle className="w-3 h-3 text-amber-500" /> Verify scan if confidence &lt; 50% or lastSeen &gt; {STALE_MIN} min</div>
        <div className="flex items-center gap-1"><Battery className="w-3 h-3 text-red-700" /> Low battery &le; {LOW_BATT}%</div>
        <div className="flex items-center gap-1"><PlugZap className="w-3 h-3 text-green-600" /> Docked/charging</div>
        <div className="flex items-center gap-1"><Lock className="w-3 h-3" /> 10‑min reservation TTL</div>
        <button className="ml-auto inline-flex items-center gap-1 text-blue-600 hover:text-blue-700"><RefreshCcw className="w-3 h-3" /> Refresh</button>
      </div>
    </div>
  );
}
"""
    with open(pro_path, "w") as f:
        f.write(ui_src)
    print("Wrote UI at", pro_path)


## Domain Expertise Guardrails & Risk Scores (display-only)

In [ ]:

CONFIG.setdefault("ethics", {"age_sex_in_decisions": False})

def mews(hr=None, rr=None, sbp=None, temp=None, avpu=None):
    s=0
    if hr is not None:
        if hr <= 40: s+=2
        elif hr <=50: s+=1
        elif hr <=100: s+=0
        elif hr <=110: s+=1
        elif hr <=130: s+=2
        else: s+=3
    if rr is not None:
        if rr <=8: s+=2
        elif rr <=14: s+=0
        elif rr <=20: s+=1
        elif rr <=29: s+=2
        else: s+=3
    if sbp is not None:
        if sbp <=70: s+=3
        elif sbp <=80: s+=2
        elif sbp <=100: s+=1
        elif sbp <=199: s+=0
        else: s+=2
    if temp is not None:
        if temp <=35.0: s+=2
        elif temp <38.5: s+=0
        else: s+=2
    if avpu is not None:
        m = {"A":0,"V":1,"P":2,"U":3}
        s += m.get(str(avpu).upper(), 0)
    return s

def qsofa(rr=None, sbp=None, ams=False):
    score = 0
    if rr is not None and rr >= 22: score += 1
    if sbp is not None and sbp <= 100: score += 1
    if bool(ams): score += 1
    return score

def sofa_lite(paO2_FiO2=None, spo2=None, platelets=None, bilirubin=None, map_val=None, pressors=False, gcs=None, creatinine=None):
    s=0
    if paO2_FiO2 is not None:
        if paO2_FiO2 < 100: s+=4
        elif paO2_FiO2 < 200: s+=3
        elif paO2_FiO2 < 300: s+=2
        elif paO2_FiO2 < 400: s+=1
    elif spo2 is not None:
        if spo2 < 85: s+=4
        elif spo2 < 90: s+=3
        elif spo2 < 92: s+=2
        elif spo2 < 94: s+=1
    if platelets is not None:
        if platelets < 20: s+=4
        elif platelets < 50: s+=3
        elif platelets < 100: s+=2
        elif platelets < 150: s+=1
    if bilirubin is not None:
        if bilirubin >= 12.0: s+=4
        elif bilirubin >= 6.0: s+=3
        elif bilirubin >= 2.0: s+=2
        elif bilirubin >= 1.2: s+=1
    if pressors: s+=3
    elif map_val is not None and map_val < 70: s+=1
    if gcs is not None:
        if gcs < 6: s+=4
        elif gcs < 10: s+=3
        elif gcs < 13: s+=2
        elif gcs < 15: s+=1
    if creatinine is not None:
        if creatinine >= 5.0: s+=4
        elif creatinine >= 3.5: s+=3
        elif creatinine >= 2.0: s+=2
        elif creatinine >= 1.2: s+=1
    return s

def heart_score(history=None, ecg=None, age=None, risk_factors=0, troponin=None):
    def hx_points(h): return { "slightly":0, "moderate":1, "highly":2 }.get(str(h).lower(), 0)
    def ecg_points(e): return { "normal":0, "nonspecific":1, "significant":2 }.get(str(e).lower(), 0)
    def age_points(a):
        if a is None: return 0
        x = int(a)
        if x < 45: return 0
        if x <= 65: return 1
        return 2
    def rf_points(n):
        try: n=int(n)
        except: n=0
        if n == 0: return 0
        if n in (1,2): return 1
        return 2
    def troponin_points(t):
        if t is None: return 0
        try: t=float(t)
        except: return 0
        if t <= 1.0: return 0
        if t <= 3.0: return 1
        return 2
    s = hx_points(history) + ecg_points(ecg) + rf_points(risk_factors) + troponin_points(troponin)
    if CONFIG.get("ethics",{}).get("age_sex_in_decisions", False):
        s += age_points(age)
    return s

def marburg_score(age=None, sex=None, known_vascular=False, patient_assumes_cardiac=False, non_palpation=True, worse_with_exercise=True):
    s = 0
    if (age is not None and sex is not None):
        try:
            a=int(age)
            if (str(sex).lower().startswith("m") and a >= 55) or (str(sex).lower().startswith("f") and a >= 65):
                s += 1
        except: pass
    if known_vascular: s+=1
    if patient_assumes_cardiac: s+=1
    if non_palpation: s+=1
    if worse_with_exercise: s+=1
    return s

def grace_proxy(age=None, hr=None, sbp=None, creatinine=None, st_deviation=False, arrest=False, killip=1, enzymes_elev=False):
    score = 0
    try: a=int(age) if age is not None else None
    except: a=None
    if a is not None:
        if a >= 80: score += 50
        elif a >= 70: score += 35
        elif a >= 60: score += 20
        elif a >= 50: score += 10
    if hr is not None:
        if hr >= 140: score += 20
        elif hr >= 100: score += 10
    if sbp is not None:
        if sbp < 80: score += 30
        elif sbp < 100: score += 20
        elif sbp < 120: score += 10
    if creatinine is not None:
        if creatinine >= 2.0: score += 20
        elif creatinine >= 1.4: score += 10
    if st_deviation: score += 30
    if arrest: score += 40
    if killip and int(killip) > 1: score += (int(killip)-1)*10
    if enzymes_elev: score += 20
    return score

def attach_scores_to_state(state):
    hr = getattr(state,"hr",None); rr = getattr(state,"rr",None); sbp = getattr(state,"sbp",None)
    temp = getattr(state,"temp_c",None); avpu = getattr(state,"avpu","A")
    ams = getattr(state,"ams",False); gcs = getattr(state,"gcs",None)
    spo2 = getattr(state,"spo2",None); creat = getattr(state,"creatinine",None)
    state.scores = {
        "MEWS": mews(hr=hr, rr=rr, sbp=sbp, temp=temp, avpu=avpu),
        "qSOFA": qsofa(rr=rr, sbp=sbp, ams=ams),
        "SOFA_lite": sofa_lite(spo2=spo2, platelets=getattr(state,"platelets",None), bilirubin=getattr(state,"bilirubin",None),
                               map_val=getattr(state,"map",None), pressors=getattr(state,"pressors",False), gcs=gcs, creatinine=creat),
        "HEART": heart_score(history=getattr(state,"hx_quality",None), ecg=getattr(state,"ecg_summary",None),
                             age=getattr(state,"age",None), risk_factors=len(getattr(state,"risk_factors",[]) or []),
                             troponin=getattr(state,"trop_t0_val",None)),
        "Marburg": marburg_score(age=getattr(state,"age",None), sex=getattr(state,"sex",None),
                                 known_vascular=getattr(state,"known_vascular",False),
                                 patient_assumes_cardiac=getattr(state,"patient_assumes_cardiac",False),
                                 non_palpation=getattr(state,"pain_not_palpable",True),
                                 worse_with_exercise=getattr(state,"worse_with_exercise",True)),
        "GRACE_proxy": grace_proxy(age=getattr(state,"age",None), hr=hr, sbp=sbp, creatinine=creat,
                                   st_deviation=(getattr(state,"ecg_summary","").lower()=="significant"),
                                   arrest=getattr(state,"arrest",False), killip=getattr(state,"killip",1),
                                   enzymes_elev=bool(getattr(state,"trop_t0_val",0)>1.0)),
    }
    return state.scores


## Patient Lingering Monitoring & Basic Needs

In [ ]:

CONFIG.setdefault("monitoring", {"lingering_hours": 2.0, "snacks_hours": 4.0})

def flag_lingering_patients(events_df, threshold_hours=None):
    import pandas as pd
    thr = float(CONFIG["monitoring"]["lingering_hours"] if threshold_hours is None else threshold_hours)
    now = pd.Timestamp.utcnow().tz_localize("UTC")
    rows=[]
    for enc, g in events_df.groupby("encounter_id"):
        g = g.copy(); g["ts"] = pd.to_datetime(g["ts"], utc=True); g = g.sort_values("ts")
        assess = g[g["event_type"].eq("ASSESSMENT")]
        last_ts = assess["ts"].iloc[-1] if len(assess) else g["ts"].iloc[-1]
        hrs = (now - last_ts).total_seconds()/3600.0
        rows.append({"encounter_id": enc, "last_assessment_ts": last_ts, "hours_since": round(hrs,1), "alert": bool(hrs >= thr)})
    import pandas as pd
    return pd.DataFrame(rows).sort_values("hours_since", ascending=False)

def skill_flag_lingering_patients(state: WorkflowState):
    last = state.event_times.get("ASSESSMENT") or state.event_times.get("TRIAGE_DONE")
    if not state.now_ts or not last: return None
    hrs = (state.now_ts - last).total_seconds()/3600.0
    thr = CONFIG["monitoring"]["lingering_hours"]
    if hrs >= thr:
        return {"action":"RECHECK_VITALS", "reason":f"Last assessment {hrs:.1f}h ago (>{thr}h)", "urgency":"high"}
    return None

def skill_offer_snacks(state: WorkflowState):
    if getattr(state,"npo", False): return None
    if getattr(state,"pressors", False): return None
    base = state.event_times.get("TRIAGE_DONE") or state.event_times.get("ARRIVAL")
    if not base or not state.now_ts: return None
    hrs = (state.now_ts - base).total_seconds()/3600.0
    thr = CONFIG["monitoring"]["snacks_hours"]
    if hrs >= thr:
        return {"action":"OFFER_SNACKS", "reason":f"Waiting {hrs:.1f}h, stable and not NPO", "urgency":"low"}
    return None


## Equipment Analytics, Finder, SOP Usage Logging

In [ ]:

class EquipmentAnalytics:
    def __init__(self): self.log=[]
    def log_scan(self, aid, loc, ts): self.log.append({"asset_id":aid,"loc":loc,"ts":ts,"evt":"scan"})
    def log_use(self, aid, ts, evt): self.log.append({"asset_id":aid,"ts":ts,"evt":f"USAGE_{evt}"})
    def frame(self):
        import pandas as pd
        return pd.DataFrame(self.log)

if 'ANALYTICS' not in globals():
    ANALYTICS = EquipmentAnalytics()

if 'ResourceTracker' in globals() and not getattr(ResourceTracker, "_analytics_patched", False):
    _prev_upd = ResourceTracker.update_from_event
    def update_from_event(self, row):
        _prev_upd(self, row)
        et = row.get("event_type","")
        ts = pd.to_datetime(row.get("ts"), utc=True)
        attrs = row.get("event_attrs","{}")
        if isinstance(attrs,str):
            try: attrs = json.loads(attrs) if attrs and attrs!="{}" else {}
            except Exception: attrs = {}
        aid = attrs.get("asset_id")
        if et=="QR_SCAN" and aid:
            ANALYTICS.log_scan(aid, row.get("location","ED"), ts)
        if et in ("ULTRASOUND_STARTED","ULTRASOUND_DONE") and aid:
            ANALYTICS.log_use(aid, ts, "start" if et.endswith("STARTED") else "done")
    ResourceTracker.update_from_event = update_from_event
    ResourceTracker._analytics_patched = True

def find_assets(query:str, types=None, max_k=10):
    q = query.lower().strip()
    hits=[]
    for aid, meta in RESOURCE.assets.items():
        if types and meta.get("type") not in set(types): continue
        blob = f"{aid} {meta.get('type','')} {meta.get('last_loc','')}".lower()
        if all(tok in blob for tok in q.split() if tok):
            hits.append({"asset_id":aid, **meta})
    return hits[:max_k]

if 'SOP_USAGE' not in globals(): SOP_USAGE = []
if 'sop_quick_open' in globals():
    _old_sop_open = sop_quick_open
    def sop_quick_open(sop_id: str):
        path = _old_sop_open(sop_id)
        if path: SOP_USAGE.append({"id": sop_id, "ts": pd.Timestamp.utcnow().isoformat()})
        return path


## Anti-Spam Gate & Fixed Follow-ups (CT 60m, Cath/Endo 30m)

In [ ]:

from collections import defaultdict
try:
    _EMIT_HISTORY
except NameError:
    _EMIT_HISTORY = defaultdict(lambda: defaultdict(list))

CONFIG.setdefault("anti_spam", {})
CFG_AS = CONFIG["anti_spam"]
CFG_AS.setdefault("cooldown_min", 60)
CFG_AS.setdefault("max_repeats_per_enc", 2)
CFG_AS.setdefault("snooze_after_accept_min", 180)

def _should_emit(enc_id: str, action: str, reason: str, now_ts: pd.Timestamp, accepted_recently: bool=False):
    hist = _EMIT_HISTORY[enc_id][(action, reason)]
    if accepted_recently and hist:
        if (now_ts - hist[-1]["ts"]).total_seconds()/60.0 < CFG_AS["snooze_after_accept_min"]:
            return False
    if hist:
        if (now_ts - hist[-1]["ts"]).total_seconds()/60.0 < CFG_AS["cooldown_min"]:
            return False
    if len(hist) >= CFG_AS["max_repeats_per_enc"]:
        return False
    return True

def _record_emit(enc_id: str, action: str, reason: str, now_ts: pd.Timestamp):
    _EMIT_HISTORY[enc_id][(action, reason)].append({"ts": now_ts})

CONFIG.setdefault("services_mobilization", {"cath_minutes": 30.0, "endo_minutes": 30.0})
def _minutes_since(ts_now, ts_then):
    if not ts_now or not ts_then: return None
    return (ts_now - ts_then).total_seconds()/60.0

def skill_follow_up_imaging_overdue(state: WorkflowState):
    ordered = state.event_times.get("ORDER_CT")
    has_result = state.event_times.get("CT_RESULT")
    if not ordered or has_result: return None
    mins = _minutes_since(state.now_ts, ordered) or 0.0
    threshold = 60.0
    if mins < threshold: return None
    action = "FOLLOW_UP_IMAGING"; reason = "CT pending > 60m"
    if not _should_emit(state.encounter_id, action, reason, state.now_ts, accepted_recently=False): return None
    _record_emit(state.encounter_id, action, reason, state.now_ts)
    return {"action": action, "reason": f"{reason} (ordered {int(mins)} min ago; threshold {int(threshold)} min)", "urgency": "medium"}

def skill_follow_up_cath_activation(state: WorkflowState):
    t_page = state.event_times.get("PAGE_CARDIOLOGY") or state.event_times.get("CATH_ACTIVATION")
    arrived = state.event_times.get("CATH_TEAM_ARRIVED") or state.event_times.get("PROCEDURE_START")
    if not t_page or arrived: return None
    mins = _minutes_since(state.now_ts, t_page) or 0.0
    threshold = float(CONFIG["services_mobilization"]["cath_minutes"])
    if mins < threshold: return None
    action = "FOLLOW_UP_CATH_ACTIVATION"; reason = f"Cath team activation pending > {int(threshold)}m"
    if not _should_emit(state.encounter_id, action, reason, state.now_ts, accepted_recently=False): return None
    _record_emit(state.encounter_id, action, reason, state.now_ts)
    return {"action": action, "reason": f"{reason} (paged {int(mins)} min ago)", "urgency": "high"}

def skill_follow_up_endo_activation(state: WorkflowState):
    t_page = state.event_times.get("PAGE_ENDOSCOPY") or state.event_times.get("ENDO_ACTIVATION")
    arrived = state.event_times.get("ENDO_TEAM_ARRIVED") or state.event_times.get("PROCEDURE_START")
    if not t_page or arrived: return None
    mins = _minutes_since(state.now_ts, t_page) or 0.0
    threshold = float(CONFIG["services_mobilization"]["endo_minutes"])
    if mins < threshold: return None
    action = "FOLLOW_UP_ENDO_ACTIVATION"; reason = f"Endoscopy activation pending > {int(threshold)}m"
    if not _should_emit(state.encounter_id, action, reason, state.now_ts, accepted_recently=False): return None
    _record_emit(state.encounter_id, action, reason, state.now_ts)
    return {"action": action, "reason": f"{reason} (paged {int(mins)} min ago)", "urgency": "high"}

if 'SKILLS' not in globals(): SKILLS = []
def _prepend_once(fn):
    if fn in SKILLS: SKILLS.remove(fn)
    SKILLS.insert(0, fn)
for fn in [skill_follow_up_imaging_overdue, skill_follow_up_cath_activation, skill_follow_up_endo_activation,
           skill_flag_lingering_patients, skill_offer_snacks]:
    _prepend_once(fn)
print("SKILLS head:", [f.__name__ for f in SKILLS[:10]])


## Domain Requirements Audit (v4.1)

In [ ]:

def domain_requirements_audit():
    checks = []
    def ok(name, cond): checks.append({"item": name, "status": "PASS" if cond else "FAIL"})
    ok("Equipment tracking core", 'ResourceTracker' in globals())
    ok("QR update via verify_scan", 'ui_apply_action' in globals())
    ok("Reservation TTL", hasattr(RESOURCE, "reserve") and hasattr(RESOURCE, "cleanup"))
    ok("Status board JSON", 'ui_status_board_json' in globals())
    ok("Equipment analytics", 'ANALYTICS' in globals())
    ok("SOP search", 'sop_search' in globals())
    ok("SOP usage logging", 'SOP_USAGE' in globals())
    ok("Lingering skill", 'skill_flag_lingering_patients' in globals())
    ok("Snacks prompt", 'skill_offer_snacks' in globals())
    for s in ["mews","qsofa","sofa_lite","heart_score","marburg_score","grace_proxy","attach_scores_to_state"]:
        ok(f"Score: {s}", s in globals())
    ok("Troponin serial", 'skill_troponin_ecg_compliance' in globals())
    ok("Troponin delta display", 'skill_review_troponin_trend' in globals())
    ok("Anti-spam gate", '_should_emit' in globals())
    ok("CT 60m follow-up", 'skill_follow_up_imaging_overdue' in globals())
    ok("Cath/Endo 30m follow-ups", 'skill_follow_up_cath_activation' in globals() and 'skill_follow_up_endo_activation' in globals())
    ok("Age/Sex not in decisions", (CONFIG.get("ethics",{}).get("age_sex_in_decisions") is False))
    import pandas as pd
    df = pd.DataFrame(checks)
    try:
        from caas_jupyter_tools import display_dataframe_to_user
        display_dataframe_to_user("Domain Requirements Audit v4.1", df)
    except Exception:
        display(df)
    return df

domain_requirements_audit();
